# PIPELINE FAST TUNING TEST — V1.3

**Nettoyage de cette session** : suppression de la section d'exploration RandomForest par défaut
et de toute la mécanique `CrossValidator`/`RandomizedSearchCV` de recherche d'hyperparamètres
(déjà trouvés, plus besoin de les rechercher) ; XGBoost et LightGBM utilisent maintenant le même
chemin "fast path" (celui qui n'a pas fait planter WSL) ; les 4 algorithmes (`RandomForest`,
`LogisticRegression`, `XGBoost`, `LightGBM`) ont chacun un checkpoint sauvegardé + un interrupteur
indépendant (section 7) pour entraîner un seul modèle sans toucher aux autres.

## 1. Imports & configuration

In [1]:
!pip install seaborn
!pip install imblearn
!pip install lightgbm
!pip install xgboost
!pip install catboost

In [2]:
from pyspark.sql import SparkSession, DataFrame
from pyspark.sql import functions as F
from pyspark.ml import Pipeline, PipelineModel
from pyspark.ml.feature import StringIndexer, OneHotEncoder, VectorAssembler, IndexToString
from pyspark.ml.classification import RandomForestClassifier
from pyspark.ml.evaluation import MulticlassClassificationEvaluator
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
# Scikit-learn classifiers for consistency (some imported locally in cells)
from lightgbm import LGBMClassifier
from xgboost import XGBClassifier
from catboost import CatBoostClassifier
# AUC-focused metrics for threshold-independent assessment
from sklearn.metrics import roc_auc_score, average_precision_score

# CORRECTIF (guide section 7.6bis) : Tomek Links ne s'applique qu'a dataset_produit
# (~140K lignes, sous-ensemble eligible, modele multi-classes) -- jamais a
# dataset_eligibilite (3M+ lignes, population complete, modele binaire traite ici).
# L'appliquer ici faisait planter le driver (OOM sur .toPandas() de 3M lignes,
# TomekLinks n'etant pas distribue). Import retire ; a reutiliser tel quel dans le
# futur notebook dataset_produit (section 6.4ter du guide), pas ici.
# from imblearn.under_sampling import TomekLinks

/usr/local/lib/python3.8/dist-packages/xgboost/core.py:265: FutureWarning: Your system has an old version of glibc (< 2.28). We will stop supporting Linux distros with glibc older than 2.28 after **May 31, 2025**. Please upgrade to a recent Linux distro (with glibc 2.28+) to use future versions of XGBoost.
Note: You have installed the 'manylinux2014' variant of XGBoost. Certain features such as GPU algorithms or federated learning are not available. To use these features, please upgrade to a recent Linux distro with glibc 2.28+, and install the 'manylinux_2_28' variant.
  warnings.warn(


In [3]:
# ═══ Bascule LOCAL / MinIO ═══
LOCAL_MODE = False  # <-- bucket MinIO complet (33 parquet)

if LOCAL_MODE:
    # Test sur un seul fichier téléchargé localement
    PATH_TRAIN_IN = "part-00000.parquet"
    PATH_TRAIN_OUT = "new_test/part-00000_final.parquet"
    PATH_SCORER_IN = None   # pas encore testé en local
    PATH_SCORER_OUT = None
    IMPUTER_MODEL_PATH = "./models/imputer_anciennete_recence"
    IMPUTER_VOLATILITE_MODEL_PATH = "./models/imputer_solde_volatilite"
    OUTLIER_BOUNDS_PATH = "./models/outlier_bounds.json"
    FLAGS_A_DROPPER_PATH = "./models/flags_extreme_a_dropper.json"
else:
    # Cluster / MinIO (bucket complet)
    # MISE A JOUR (suite a EDA_advanced_eligibilite v3, corrigee) : on pointe maintenant
    # sur le dataset enrichi v3 (ratios, colonne d'interaction, colonnes binnees,
    # CODE_VILLE regroupe) plutot que sur dataset_eligibilite_final/ brut -- cf.
    # EDA_advanced_eligibilite_v3, section 11 ("prochaine etape").
    PATH_TRAIN_IN = "s3a://processed-data/dataset_eligibilite_features_v3/"
    PATH_TRAIN_OUT = "s3a://processed-data/dataset_eligibilite_features_v3/"
    # Pas encore de dataset a scorer separe pour l'eligibilite -- laisse a None
    # (meme convention que la branche LOCAL_MODE=True) jusqu'a ce qu'il existe sur MinIO.
    PATH_SCORER_IN = None
    PATH_SCORER_OUT = None
    IMPUTER_MODEL_PATH = "s3a://ml-scoring/models/imputer_anciennete_recence"
    IMPUTER_VOLATILITE_MODEL_PATH = "s3a://ml-scoring/models/imputer_solde_volatilite"
    OUTLIER_BOUNDS_PATH = "s3a://ml-scoring/models/outlier_bounds/"
    FLAGS_A_DROPPER_PATH = "s3a://ml-scoring/models/flags_extreme_a_dropper/"
    # Config features v3, ecrite par EDA_advanced_eligibilite_v3 section 9 -- source
    # de verite pour les listes de la cellule suivante (au lieu de les dupliquer a la
    # main comme avant, cf. le TODO "factoriser dans un module partage").
    FEATURE_CONFIG_IN = "s3a://ml-scoring/models/feature_config_v3.json"


In [4]:
# MISE A JOUR : ces listes viennent de feature_config_v3.json (EDA_advanced_eligibilite_v3,
# section 9), APRES correction du bug de section 8 qui droppait a tort flux_cred_total
# (IV=0.417, la feature #1) et deux autres features IV>=0.1 (montant_total_retraits,
# montant_total_payfac) a cause d'une convention de nommage coincidentielle avec leur
# partenaire de correlation "nb_*". Cf. EDA_advanced_eligibilite_v3_FIXED.ipynb, cellule
# "Ratios pour les paires redondantes" -- le garde-fou IV>=0.1 les garde desormais.
#
#
# TODO V2 (toujours valable) : charger dynamiquement FEATURE_CONFIG_IN au lieu de dupliquer
# ces valeurs ici -- ex. json.loads(\"\\n\".join(spark.sparkContext.textFile(FEATURE_CONFIG_IN).collect()))
# une fois la session Spark ouverte (cellule suivante). Duplique ici pour rester lisible/
# debuggable sans dependre d'un objet Spark des la config.

COLS_CATEGORIELLES_BASSE_CARDINALITE = [
    "CUSTOMER_RATING", "pack_actuel", "MARITAL_STATUS", "NOMBRE_ENFANT", "pack_etat",
]

# CORRECTIF (deja en place) : GENDER (IV=0.0015), TAILLE_ENTREPRI (IV=0.0006) et BPR
# (IV=0.017) sont retires -- sous le seuil IV>=0.02 (bruit, pas signal). NOMBRE_ENFANT
# (IV=0.040) est ajoute -- n'etait utilise nulle part avant (ni categorielle ni numerique).

COL_HAUTE_CARDINALITE = "CODE_VILLE_regroupe"  # version regroupee (petites villes -> "AUTRE"), plus fiable que CODE_VILLE brut -- cf. EDA v3 section 1bis
COL_LABEL = "label_epargne"
COLS_A_EXCLURE_DES_FEATURES = ["label_code", "label_epargne", "label_nom", "RADICAL"]  # cible (2 formes) + identifiants

# --- Nouveau (EDA v3, section 8) : colonnes categorielles derivees, encodees comme les
#     basse-cardinalite (StringIndexer + OneHotEncoder, cellule "Construction du pipeline")
#     plutot que comme des numeriques -- ce sont des chaines de caracteres. ---

## 2. Spark session & chargement du train nettoyé

In [5]:
from pyspark.sql import SparkSession, DataFrame
from pyspark.sql import functions as F


def get_spark() -> SparkSession:
    if LOCAL_MODE:
        builder = (
            SparkSession.builder
            .master("local[*]")
            .appName("training_pipeline_scoring")
            .config("spark.driver.memory", "4g")
        )
    else:
        builder = (
            SparkSession.builder
            .appName("training_pipeline_scoring")
            .master("spark://spark-master:7077")

            # Memory
            .config("spark.driver.memory", "6g")
            .config("spark.executor.memory", "2g")

            # Executor resources
            .config("spark.executor.cores", "2")

            # CORRECTIF (crash log : "Removing executor 0 with no recent heartbeats" /
            # "worker lost: Not receiving heartbeat for 60 seconds") -- la RandomizedSearchCV
            # (section 9quater fast path) monopolise le CPU du driver pendant la recherche
            # d'hyperparametres, les executors n'ont plus le temps d'envoyer leur heartbeat
            # dans la fenetre par defaut (120s / 10s). On l'allonge.
            .config("spark.network.timeout", "600s")
            .config("spark.executor.heartbeatInterval", "60s")

            # Shuffle
            .config("spark.sql.shuffle.partitions", "12")

            # CORRECTIF (403 Forbidden sur s3a://processed-data/... ) -- les variables
            # SPARK_HADOOP_FS_S3A_* du docker-compose (image spark-custom:3.5.1) ne sont
            # visiblement pas traduites en config Hadoop reelle : le diagnostic montrait
            # fs.s3a.endpoint=s3.amazonaws.com et path.style.access=false, donc Spark
            # essayait de parler au vrai AWS S3 avec les identifiants MinIO. On force la
            # config S3A explicitement ici, alignee sur le docker-compose (service minio,
            # port 9000, minioadmin/minioadmin123, MinIO = path-style, HTTP sans TLS).
            .config("spark.hadoop.fs.s3a.endpoint", "http://minio:9000")
            .config("spark.hadoop.fs.s3a.access.key", "minioadmin")
            .config("spark.hadoop.fs.s3a.secret.key", "minioadmin123")
            .config("spark.hadoop.fs.s3a.path.style.access", "true")
            .config("spark.hadoop.fs.s3a.connection.ssl.enabled", "false")
            .config("spark.hadoop.fs.s3a.impl", "org.apache.hadoop.fs.s3a.S3AFileSystem")
        )

    spark = builder.getOrCreate()
    spark.sparkContext.setLogLevel("WARN")
    return spark


def charger_dataset(path: str) -> DataFrame:
    if path.endswith(".csv"):
        df = (
            spark.read
            .option("header", True)
            .option("inferSchema", True)
            .option("samplingRatio", "0.1")
            .csv(path)
        )
        df = df.withColumn(
            COL_HAUTE_CARDINALITE,
            F.col(COL_HAUTE_CARDINALITE).cast("string")
        )
    else:
        df = spark.read.parquet(path)

    # CORRECTIF (v1.9) : le cast en double ci-dessous datait d'une ancienne version de
    # interaction_solde_min_x_depot_moyen qui avait 1 663 638 valeurs distinctes (un
    # concat brut de 2 colonnes continues -- quasi un identifiant, pas une categorie ;
    # le OHE aurait explose a ~1,6M colonnes). L'EDA binne desormais f1/f2 en quartiles
    # AVANT de les combiner (4x4 = 16 cellules max) et ecrit deja la colonne comme
    # string cote parquet -- c'est une vraie categorielle basse-cardinalite maintenant,
    # exactement ce que la config (feature_config_v3.json, cols_categorielles_basse_cardinalite)
    # declare. La caster en double ici l'aurait re-transformee en variable ordinale brute
    # (0 a 15) au lieu de la laisser passer par StringIndexer + OneHotEncoder -- perte de
    # signal silencieuse (les 16 cellules d'une grille 4x4 n'ont pas de relation d'ordre
    # lineaire). On ne cast plus : on laisse le dtype du parquet (string) tel quel, et la
    # cellule "chargement dynamique de feature_config_v3.json" (juste apres) se charge de
    # la router vers les categorielles.

    return df


spark = get_spark()
df_train_full = charger_dataset(PATH_TRAIN_IN)
print(df_train_full.count())
df_train_full.printSchema()

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/08/13 14:43:42 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
26/08/13 14:43:46 WARN MetricsConfig: Cannot locate configuration: tried hadoop-metrics2-s3a-file-system.properties,hadoop-metrics2.properties
26/08/13 14:44:03 WARN TaskSchedulerImpl: Initial job has not accepted any resources; check your cluster UI to ensure that workers are registered and have sufficient resources
26/08/13 14:44:18 WARN TaskSchedulerImpl: Initial job has not accepted any resources; check your cluster UI to ensure that workers are registered and have sufficient resources
[Stage 1:=======================================>                   (2 + 1) / 3]

3172296
root
 |-- CODE_VILLE: string (nullable = true)
 |-- BPR: string (nullable = true)
 |-- GENDER: string (nullable = true)
 |-- MARITAL_STATUS: string (nullable = true)
 |-- NOMBRE_ENFANT: string (nullable = true)
 |-- CUSTOMER_RATING: string (nullable = true)
 |-- TAILLE_ENTREPRI: string (nullable = true)
 |-- age_client: long (nullable = true)
 |-- label_code: string (nullable = true)
 |-- label_nom: string (nullable = true)
 |-- pack_actuel: string (nullable = true)
 |-- pack_etat: string (nullable = true)
 |-- solde_moyen: double (nullable = true)
 |-- solde_min: double (nullable = true)
 |-- nb_mois_observes_solde: long (nullable = true)
 |-- depot_moyen: double (nullable = true)
 |-- flux_cred_total: double (nullable = true)
 |-- nb_mois_avec_flux: long (nullable = true)
 |-- nb_operations_gab: long (nullable = true)
 |-- montant_total_gab: double (nullable = true)
 |-- montant_moyen_gab: double (nullable = true)
 |-- nb_retraits: long (nullable = true)
 |-- montant_total_re

In [6]:
# ═══ Chargement dynamique de feature_config_v3.json (au lieu de la copie figée cellule 5) ═══
# BUG POTENTIEL COUVERT ICI : la colonne d'interaction (COLONNE_INTERACTION, cellule 5) est
# nommée dynamiquement côté EDA -- f"interaction_{f1}_x_{f2}" où (f1, f2) est LA PAIRE LA PLUS
# FORTE au moment du run EDA (section 5 du notebook EDA, TOP_N_INTERACTIONS). Si les nouvelles
# features de l'EDA changent le classement, cette paire -- et donc le nom de la colonne -- peut
# changer d'un run EDA à l'autre. Le pipeline pointait vers l'ancien nom en dur
# ("interaction_solde_min_x_depot_moyen") ; si l'EDA a produit un nom différent, cette colonne
# n'existe simplement plus dans df_train_full, le cast en double (charger_dataset) est skippé
# silencieusement (garde-fou `if COLONNE_INTERACTION in df.columns`), et colonnes_features_numeriques()
# ne la détecte pas non plus puisqu'elle n'est pas dans le DataFrame -- la feature d'interaction
# la plus forte disparaît du feature set SANS AUCUNE ERREUR. Pareil pour les listes catégorielles/
# numériques/binnées de la cellule 5 : recopiées à la main, donc périmées dès que l'EDA change
# quoi que ce soit tant que quelqu'un ne les remet pas à jour manuellement.
#
# Ici, on lit feature_config_v3.json (écrit par l'EDA, section 9) et on ÉCRASE les variables de
# la cellule 5 avec les valeurs réelles du dernier run EDA. Si le chargement échoue (config
# absente, EDA jamais lancée), on retombe sur les valeurs codées en dur de la cellule 5 --
# mais avec un avertissement bien visible plutôt qu'un silence.
import json as _json_cfg

_config_charge = False
try:
    _config_json_str = "\n".join(spark.sparkContext.textFile(FEATURE_CONFIG_IN).collect())
    _cfg = _json_cfg.loads(_config_json_str)

    _ancienne_col_interaction = COLONNE_INTERACTION
    COLS_CATEGORIELLES_BASSE_CARDINALITE = [
        c for c in _cfg["cols_categorielles_basse_cardinalite"]
        if c not in _cfg.get("colonnes_binnees", []) and c != _cfg.get("colonne_interaction")
    ]
    for new_c in ["pack_actuel_x_CUSTOMER_RATING", "pack_etat_x_CUSTOMER_RATING", "pack_actuel_x_pack_etat"]:
        if new_c not in COLS_CATEGORIELLES_BASSE_CARDINALITE:
            COLS_CATEGORIELLES_BASSE_CARDINALITE.append(new_c)
    COL_HAUTE_CARDINALITE = _cfg["col_haute_cardinalite"]
    COLONNES_BINNEES = _cfg.get("colonnes_binnees", [])
    COLONNE_INTERACTION = _cfg.get("colonne_interaction")
    COLS_NUMERIQUES_RATIOS_V3 = _cfg.get("cols_numeriques_ratios_ajoutees", [])
    _cols_numeriques_config = _cfg.get("cols_numeriques", [])  # traçabilité / diagnostic uniquement

    _config_charge = True
    print(f"feature_config_v3.json chargé depuis {FEATURE_CONFIG_IN} -- config du dernier run EDA appliquée.")
    if _ancienne_col_interaction != COLONNE_INTERACTION:
        print(f"  ATTENTION -- colonne d'interaction différente de la copie en dur cellule 5 : "
              f"'{_ancienne_col_interaction}' -> '{COLONNE_INTERACTION}'. C'est attendu si l'EDA a "
              f"trouvé une paire différente comme interaction la plus forte -- sans ce chargement "
              f"dynamique, l'ancienne colonne (qui n'existe plus dans le dataset) aurait été utilisée "
              f"silencieusement en moins, sans erreur.")
    if _cfg.get("features_suspectes_iv_superieur_0_5"):
        print(f"  ATTENTION -- features avec IV > 0.5 signalées par l'EDA comme suspectes de fuite : "
              f"{_cfg['features_suspectes_iv_superieur_0_5']} -- à vérifier avant d'entraîner dessus.")
except Exception as e:
    print(f"(feature_config_v3.json non chargé -- {type(e).__name__}: {e}. "
          f"On retombe sur la copie codée en dur de la cellule 5 -- pensez à la mettre à jour "
          f"manuellement si l'EDA a changé les features.)")


(feature_config_v3.json non chargé -- NameError: name 'COLONNE_INTERACTION' is not defined. On retombe sur la copie codée en dur de la cellule 5 -- pensez à la mettre à jour manuellement si l'EDA a changé les features.)


In [7]:
def colonnes_features_numeriques(df: DataFrame) -> list:
    """Toutes les colonnes numériques du dataset, hors cible et hors colonnes
    catégorielles couvertes par l'encodage (section 6) -- sinon on se retrouverait
    avec CODE_VILLE brut ET CODE_VILLE_idx dans le vecteur final. Utilisée à la
    fois pour Tomek Links (section 3, sur les features numériques uniquement --
    TomekLinks calcule des distances, incompatible avec des catégorielles brutes
    non encodées à ce stade) et pour VectorAssembler (section 6)."""
    cols_categorielles_brutes = set(COLS_CATEGORIELLES_BASSE_CARDINALITE + [COL_HAUTE_CARDINALITE])
    return [
        c for c, t in df.dtypes
        if t in ("int", "bigint", "double", "float")
        and c not in COLS_A_EXCLURE_DES_FEATURES
        and c not in cols_categorielles_brutes
    ]


feature_cols_numeriques = colonnes_features_numeriques(df_train_full)
print(f"Features numériques ({len(feature_cols_numeriques)}) : {feature_cols_numeriques}")


Features numériques (40) : ['age_client', 'solde_moyen', 'solde_min', 'nb_mois_observes_solde', 'depot_moyen', 'flux_cred_total', 'nb_mois_avec_flux', 'nb_operations_gab', 'montant_total_gab', 'montant_moyen_gab', 'nb_retraits', 'montant_total_retraits', 'nb_paiements_digitaux', 'montant_total_payfac', 'nb_vignettes_payees', 'montant_total_vignette', 'label_eligibilite', 'jamais_active_digital', 'jamais_utilise_gab', 'anciennete_digitale_jours_imp', 'recence_gab_jours_imp', 'nb_mois_observes_solde_etait_extreme', 'solde_moyen_etait_extreme', 'solde_min_etait_extreme', 'solde_max_etait_extreme', 'depot_moyen_etait_extreme', 'flux_cred_moyen_etait_extreme', 'flux_cred_total_etait_extreme', 'montant_total_gab_etait_extreme', 'montant_moyen_gab_etait_extreme', 'montant_total_retraits_etait_extreme', 'montant_total_payfac_etait_extreme', 'montant_total_vignette_etait_extreme', 'solde_volatilite_indefinie', 'solde_volatilite_relative_imp', 'montant_total_retraits_moyen', 'montant_total_payfa

In [10]:
COL_LABEL = "solde_moyen"
df_train_full.select(COL_LABEL).describe().show()

+-------+-------------------+
|summary|        solde_moyen|
+-------+-------------------+
|  count|            3172287|
|   mean|  7329.721797455981|
| stddev| 11198.566854071314|
|    min|-17819.264166666668|
|    max| 29468.942499999997|
+-------+-------------------+



In [11]:
# Create épargne product target column for scoring
# label_epargne = 1 if label_nom == "EPARGNE EVOLUTION", 0 otherwise
from pyspark.sql import functions as F

# Add the épargne target column
df_train_full = df_train_full.withColumn(
    "label_epargne",
    F.when(F.col("label_nom") == "EPARGNE EVOLUTION", 1).otherwise(0)
)

# Show distribution of the new target
print("Distribution of label_epargne (épárgne product target):")
df_train_full.groupBy("label_epargne").count().show()

# Also show relationship with original label_eligibilite for reference
print("Joint distribution of label_eligibilite and label_epargne:")
df_train_full.groupBy("label_eligibilite", "label_epargne").count().show()

Distribution of label_epargne (épárgne product target):


+-------------+-------+
|label_epargne|  count|
+-------------+-------+
|            0|3162934|
|            1|   9362|
+-------------+-------+

Joint distribution of label_eligibilite and label_epargne:


[Stage 11:======================================>                   (2 + 1) / 3]

+-----------------+-------------+-------+
|label_eligibilite|label_epargne|  count|
+-----------------+-------------+-------+
|                1|            1|   9362|
|                1|            0| 125565|
|                0|            0|3037369|
+-----------------+-------------+-------+



## 3. Nettoyage de frontiere (Tomek Links) -- non applicable ici, volontairement desactive

Tomek Links (guide section 7.6bis) ne concerne que le modele **produit**
(`dataset_produit`, sous-ensemble eligible, ~140K lignes, multi-classes) -- jamais le
modele **eligibilite** traite dans ce notebook (`dataset_eligibilite`, population
complete, 3M+ lignes). L'appliquer sur la totalite provoquait un `OutOfMemoryError`
cote driver : `TomekLinks` (imbalanced-learn) n'est pas distribue, `.toPandas()` sur
3M lignes charge tout en memoire dans un seul process Python.

Le desequilibre du modele d'eligibilite (~4,6% de positifs) est traite plus loin,
avec des outils adaptes a ce volume : ponderation par classe (section 5, formule
adoucie), et reglage du seuil de decision pour XGBoost/LightGBM (section 9quater bis).
La cellule suivante ne fait qu'un diagnostic de distribution -- `df_train_full`
n'est pas modifie.


In [12]:
# Diagnostic uniquement -- df_train_full n'est PAS modifie ici (cf. section 3 ci-dessus).
total_avant = df_train_full.count()
print(f"Lignes (population complete, avant tout traitement de desequilibre) : {total_avant}")
df_train_full.groupBy(COL_LABEL).count().withColumn(
    "part", F.round(F.col("count") / total_avant, 4)
).orderBy(COL_LABEL).show()


Lignes (population complete, avant tout traitement de desequilibre) : 3172296


[Stage 19:>                                                         (0 + 2) / 2]

+-------------------+-----+------+
|        solde_moyen|count|  part|
+-------------------+-----+------+
|               NULL|    9|   0.0|
|-17819.264166666668| 2842|9.0E-4|
|-17812.379999999997|    1|   0.0|
|          -17804.62|    1|   0.0|
|       -17800.96875|    1|   0.0|
| -17800.66947916667|    1|   0.0|
|-17800.657083333335|    1|   0.0|
|-17799.316250000003|    1|   0.0|
|-17796.623333333333|    1|   0.0|
|-17782.136666666665|    1|   0.0|
|       -17778.71625|    1|   0.0|
|          -17777.47|    1|   0.0|
|        -17777.4025|    1|   0.0|
|-17773.731666666667|    1|   0.0|
| -17772.20916666667|    1|   0.0|
|-17768.817499999997|    1|   0.0|
|-17766.412916666664|    1|   0.0|
|       -17765.34625|    1|   0.0|
|-17764.632499999996|    1|   0.0|
|         -17763.915|    1|   0.0|
+-------------------+-----+------+
only showing top 20 rows



## 4. Split train / validation

Un split est fait ici **en plus** du train/scoring déjà séparé en amont (Partie 1 EDA) : celui-là
sépare "données qu'on a le droit d'utiliser" de "données de production", celui-ci sépare "données
pour entraîner" de "données pour évaluer honnêtement avant de livrer le modèle". Sans ce split, la
métrique d'évaluation serait mesurée sur les données mêmes qui ont servi à fitter les indexeurs et
le classifieur -- optimiste, pas fiable.

Split fait **après** Tomek Links (section 3), sur `df_train_full` déjà nettoyé -- le `PipelineModel`
final (section 10) sera de toute façon refit sur 100% de ce `df_train_full` nettoyé une fois la
V1.1 validée ici.

In [15]:
RANDOM_SEED = 42

df_fit, df_val = df_train_full.randomSplit([0.8, 0.2], seed=RANDOM_SEED)
df_fit.cache()
df_val.cache()
print(f"Fit  : {df_fit.count()} lignes")
print(f"Val  : {df_val.count()} lignes")

print("\nÉquilibre des classes -- fit vs val (doivent être comparables) :")
df_fit.groupBy(COL_LABEL).count().withColumn("part_fit", F.round(F.col("count") / df_fit.count(), 3)).show()
df_val.groupBy(COL_LABEL).count().withColumn("part_val", F.round(F.col("count") / df_val.count(), 3)).show()


26/08/13 14:50:36 WARN CacheManager: Asked to cache already cached data.
26/08/13 14:50:36 WARN CacheManager: Asked to cache already cached data.


Fit  : 2538221 lignes
Val  : 634075 lignes

Équilibre des classes -- fit vs val (doivent être comparables) :


+-------------------+-----+--------+
|        solde_moyen|count|part_fit|
+-------------------+-----+--------+
| -14.81857142857143|    1|     0.0|
|          8438.1325|    1|     0.0|
|  4850.255416666667|    1|     0.0|
| 1428.1625000000004|    1|     0.0|
| -670.7833333333332|    1|     0.0|
|  8757.755416666665|    1|     0.0|
| -473.6999999999999|    1|     0.0|
|-1744.7370833333334|    1|     0.0|
|          19223.945|    1|     0.0|
| 18308.557083333333|    1|     0.0|
| 455.12833333333333|    1|     0.0|
|  858.9833333333331|    1|     0.0|
|  2212.684166666667|    1|     0.0|
|  2595.247708333334|    1|     0.0|
|-205.17000000000004|   29|     0.0|
|  6672.835000000002|    1|     0.0|
| 10221.686666666668|    1|     0.0|
| 12927.938541666665|    1|     0.0|
| 1804.1929166666669|    1|     0.0|
|  4528.927083333332|    1|     0.0|
+-------------------+-----+--------+
only showing top 20 rows



+-------------------+-----+--------+
|        solde_moyen|count|part_val|
+-------------------+-----+--------+
|        -1665.33375|    1|     0.0|
|-162.83999999999997| 4311|   0.007|
| 12092.069583333336|    1|     0.0|
|-244.44458333333333|    1|     0.0|
|  2718.807916666667|    1|     0.0|
|  3816.678260869565|    1|     0.0|
|              80.72|    1|     0.0|
|  65.21000000000001|    1|     0.0|
|-237.86291666666668|    1|     0.0|
|  7633.918750000001|    1|     0.0|
|               58.0|   49|     0.0|
|  85.81708333333331|    1|     0.0|
| 2.7866666666666666|    1|     0.0|
| 335.71259259259256|    1|     0.0|
| 149.94083333333333|    1|     0.0|
|-112.91958333333336|    1|     0.0|
|  26649.40958333334|    1|     0.0|
| -779.2733333333334|    1|     0.0|
|  4186.062083333334|    1|     0.0|
| 13559.792083333336|    1|     0.0|
+-------------------+-----+--------+
only showing top 20 rows



## 5. Pondération inverse-fréquence par classe

Guide, section 7.6. Complémentaire à Tomek Links (section 3) : Tomek nettoie les lignes
ambiguës à la frontière, la pondération force le classifieur à ne pas ignorer les classes rares
pendant l'apprentissage. **Calculée après Tomek Links**, sur `df_fit` -- la distribution des
classes a changé suite au nettoyage de frontière, un poids calculé sur la distribution brute
d'avant-Tomek serait faux.

Formule (guide) : `poids_classe = total / (nb_classes × effectif_de_la_classe)` -- pondération
inverse-fréquence standard multi-classe, chaque classe rare reçoit un poids proportionnellement
plus élevé.

In [16]:
def calculer_poids_classe(df: DataFrame, col_label: str = COL_LABEL) -> DataFrame:
    """Retourne un DataFrame [col_label, poids_classe] a joindre sur le train
    (fit ou full) juste avant l'entrainement. Recalcule a chaque fois que la
    population d'entree change (ex. fit vs. full au refit de la section 10).

    CORRECTIF (guide section 7.6quater, point 2) : formule adoucie en racine
    carree plutot que la ponderation inverse-frequence brute. Sur un desequilibre
    aussi marque (~4,6% de positifs), la formule brute sur-corrigeait et poussait
    RandomForest/LogReg/DecisionTree a sur-predire la classe rare (rappel correct
    mais precision ~8-9%). La racine carree garde la classe rare sur-ponderee,
    mais moins agressivement.
    """
    effectifs = df.groupBy(col_label).count()
    total = df.count()
    nb_classes = effectifs.count()

    poids = effectifs.withColumn(
        "poids_classe", F.sqrt(total / (nb_classes * F.col("count")))
    ).select(col_label, "poids_classe")

    print(f"\nPoids par classe (total={total}, nb_classes={nb_classes}, formule=sqrt) :")
    poids.orderBy(col_label).show()
    return poids


poids_par_classe_fit = calculer_poids_classe(df_fit)

# CORRECTIF (bug "Lignes ecrites != lignes attendues" dans vers_pandas_xy, section 9) :
# ce join ecrase df_fit par un NOUVEAU DataFrame non cache -- le .cache() de la
# cellule 13 ne portait que sur l'ancien df_fit (avant join), qui n'est plus
# reference par ce nom. Sans cache ici, chaque action Spark en aval (df.count(),
# puis le passage separe dans vers_pandas_xy) recalcule toute la lignee
# randomSplit -> join depuis la source, et deux recalculs independants d'une
# lignee avec shuffle peuvent legerement diverger (retry de tache, AQE, etc.),
# d'ou l'ecart observe entre count() et les lignes reellement collectees.
df_fit = df_fit.join(poids_par_classe_fit, on=COL_LABEL).cache()
df_fit.count()  # materialise le cache immediatement, une seule fois pour toutes les actions suivantes


Poids par classe (total=2538221, nb_classes=1878043, formule=sqrt) :


+-------------------+--------------------+
|        solde_moyen|        poids_classe|
+-------------------+--------------------+
|               NULL| 0.41102378772617326|
|-17819.264166666668|0.024422043352403815|
|-17812.379999999997|  1.1625508301206287|
| -17800.66947916667|  1.1625508301206287|
|-17800.657083333335|  1.1625508301206287|
|-17799.316250000003|  1.1625508301206287|
|-17796.623333333333|  1.1625508301206287|
|          -17777.47|  1.1625508301206287|
|        -17777.4025|  1.1625508301206287|
|-17773.731666666667|  1.1625508301206287|
| -17772.20916666667|  1.1625508301206287|
|-17766.412916666664|  1.1625508301206287|
|       -17765.34625|  1.1625508301206287|
|-17764.632499999996|  1.1625508301206287|
|         -17763.915|  1.1625508301206287|
|-17760.126249999998|  1.1625508301206287|
|-17756.953333333335|  1.1625508301206287|
|         -17753.785|  1.1625508301206287|
|-17751.990416666664|  1.1625508301206287|
|-17751.906190476187|  1.1625508301206287|
+----------

2538213

## 6. Construction du pipeline

Trois familles de stages, dans l'ordre où Spark doit les exécuter :

1. **Encodage catégoriel** : `StringIndexer` + `OneHotEncoder` pour les colonnes basse-cardinalité,
   `StringIndexer` seul pour `CODE_VILLE` (haute cardinalité — un `OneHotEncoder` dessus
   exploserait le nombre de features pour un gain incertain).
2. **Indexation de la cible** : `label_nom` (string) → `label_idx` (double), requis par les
   classifieurs MLlib.
3. **Assemblage** : toutes les features numériques + toutes les sorties d'encodage dans un seul
   vecteur `features`, puis le classifieur, **pondéré par `poids_classe`** (section 5).

`RandomForestClassifier` choisi pour cette V1.1 : robuste aux features non standardisées, gère
nativement `CODE_VILLE_idx` à haute cardinalité sans que l'ordre numérique arbitraire de l'index
ne biaise le modèle (contrairement à une régression logistique), **et supporte `weightCol`
nativement depuis Spark 3.0** — exactement l'exemple du guide (section 7.5).

In [19]:
COLONNES_BINNEES = []
COLONNE_INTERACTION = []

def construire_stages_encodage(cols_basse_cardinalite: list, col_haute_cardinalite: str):
    indexers = [
        StringIndexer(inputCol=c, outputCol=f"{c}_idx", handleInvalid="keep")
        for c in cols_basse_cardinalite
    ]
    encoders = [
        OneHotEncoder(inputCol=f"{c}_idx", outputCol=f"{c}_ohe")
        for c in cols_basse_cardinalite
    ]
    indexer_haute_card = StringIndexer(
        inputCol=col_haute_cardinalite, outputCol=f"{col_haute_cardinalite}_idx", handleInvalid="keep"
    )
    return indexers + encoders + [indexer_haute_card]


# MISE A JOUR : les colonnes binnees (EDA v3 section 2/8) et la colonne d'interaction
# (EDA v3 section 5) sont des chaines de caracteres derivees -- meme traitement que les
# categorielles basse-cardinalite (StringIndexer + OneHotEncoder), pas assemblees telles
# quelles. On les ajoute donc a la liste passee a construire_stages_encodage().
# APRÈS :
# cellule 17 -- APRÈS :
# CORRECTIF : COLONNES_BINNEES (cellule 5) est une copie figee de feature_config_v3.json,
# qui peut lister des colonnes _bin qui n'ont en realite jamais ete creees cote EDA
# (colonnes quasi-constantes silencieusement "continue"-ees -- cf. EDA_advanced_eligibilite_v3
# section 8, bug corrige separement). Plutot que de dependre d'une liste re-copiee a la main
# a chaque run EDA, on filtre ici contre le schema reellement charge -- robuste peu importe
# ce que le prochain run EDA produit.
colonnes_binnees_disponibles = [c for c in COLONNES_BINNEES if c in df_train_full.columns]
colonnes_binnees_absentes = [c for c in COLONNES_BINNEES if c not in df_train_full.columns]
if colonnes_binnees_absentes:
    print(f"(colonnes binnées listées dans la config mais absentes du dataset chargé -- "
          f"ignorées : {colonnes_binnees_absentes})")

cols_categorielles_completes = COLS_CATEGORIELLES_BASSE_CARDINALITE + colonnes_binnees_disponibles

# CORRECTIF (v1.10) : le bloc precedent supposait que COLONNE_INTERACTION etait deja
# castee en double au chargement (charger_dataset) et donc deja recuperee cote numerique
# -- c'etait vrai avant le correctif v1.9, qui a justement retire ce cast (la colonne est
# une categorie a 16 valeurs, pas une variable continue -- cf. commentaire charger_dataset).
# Depuis v1.9 elle reste "string" dans le DataFrame Spark : l'ancien bloc l'ajoutait donc
# a feature_cols_numeriques encore vide de traitement, et VectorAssembler recevait une
# colonne string brute -> IllegalArgumentException ("Data type string ... is not
# supported"). Fix : meme traitement que les autres categorielles basse-cardinalite --
# StringIndexer + OneHotEncoder -- pas d'ajout a feature_cols_numeriques.
if COLONNE_INTERACTION and COLONNE_INTERACTION not in cols_categorielles_completes:
    cols_categorielles_completes = cols_categorielles_completes + [COLONNE_INTERACTION]

encodage_stages = construire_stages_encodage(cols_categorielles_completes, COL_HAUTE_CARDINALITE)

label_indexer = StringIndexer(inputCol=COL_LABEL, outputCol="label_idx", handleInvalid="error")

feature_cols_encodees = [f"{c}_ohe" for c in cols_categorielles_completes] + [f"{COL_HAUTE_CARDINALITE}_idx"]
feature_cols = feature_cols_numeriques + feature_cols_encodees

print(f"Features numériques ({len(feature_cols_numeriques)}) : {feature_cols_numeriques}")
print(f"Features encodées   ({len(feature_cols_encodees)}) : {feature_cols_encodees}")

# handleInvalid="skip" : "poids_classe" n'est PAS dans inputCols, VectorAssembler
# ne le touche pas -- il reste disponible comme colonne à part pour weightCol ci-dessous.
assembler = VectorAssembler(inputCols=feature_cols, outputCol="features", handleInvalid="keep")

# CORRECTIF : RandomForestClassifier traite CODE_VILLE_idx comme une feature
# catégorielle (métadonnée posée par le StringIndexer) et exige maxBins >= son
# nombre de modalités -- sinon Spark lève IllegalArgumentException au .fit().
# maxBins par défaut (32) est très inférieur aux modalités de CODE_VILLE_regroupe
# (guide / COL_HAUTE_CARDINALITE) : on le dimensionne dynamiquement plutôt que
# de fixer une valeur en dur qui casserait si la cardinalité évolue.
nb_modalites_ville = df_train_full.select(COL_HAUTE_CARDINALITE).distinct().count()
max_bins = max(32, nb_modalites_ville + 1)  # +1 : marge pour la modalité "inconnue" (handleInvalid="keep")
print(f"{COL_HAUTE_CARDINALITE} : {nb_modalites_ville} modalités observées -> maxBins={max_bins}")

clf = RandomForestClassifier(
    labelCol="label_idx",
    featuresCol="features",
    weightCol="poids_classe",
    numTrees=30,
    maxDepth=5,
    maxBins=max_bins,
    seed=RANDOM_SEED,
)

pipeline = Pipeline(stages=encodage_stages + [label_indexer, assembler, clf])


Features numériques (40) : ['age_client', 'solde_moyen', 'solde_min', 'nb_mois_observes_solde', 'depot_moyen', 'flux_cred_total', 'nb_mois_avec_flux', 'nb_operations_gab', 'montant_total_gab', 'montant_moyen_gab', 'nb_retraits', 'montant_total_retraits', 'nb_paiements_digitaux', 'montant_total_payfac', 'nb_vignettes_payees', 'montant_total_vignette', 'label_eligibilite', 'jamais_active_digital', 'jamais_utilise_gab', 'anciennete_digitale_jours_imp', 'recence_gab_jours_imp', 'nb_mois_observes_solde_etait_extreme', 'solde_moyen_etait_extreme', 'solde_min_etait_extreme', 'solde_max_etait_extreme', 'depot_moyen_etait_extreme', 'flux_cred_moyen_etait_extreme', 'flux_cred_total_etait_extreme', 'montant_total_gab_etait_extreme', 'montant_moyen_gab_etait_extreme', 'montant_total_retraits_etait_extreme', 'montant_total_payfac_etait_extreme', 'montant_total_vignette_etait_extreme', 'solde_volatilite_indefinie', 'solde_volatilite_relative_imp', 'montant_total_retraits_moyen', 'montant_total_payfa

## 7. Interrupteurs d'entraînement — reload vs retrain, par algorithme

Chaque algorithme a son propre interrupteur. `True` = (ré)entraîne et écrase le checkpoint.
`False` = recharge le modèle déjà sauvegardé, aucun `.fit()` n'est déclenché. Ils sont
indépendants : pour ne tester que LightGBM par exemple, laissez les trois autres à `False`.

**Sections supprimées dans cette version** (V1.2 → V1.3, nettoyage) : l'ancienne section 7-9
(un 3ᵉ `RandomForest` d'exploration, hyperparamètres par défaut) et toute la mécanique
`CrossValidator`/`algos_config`/`entrainer_un_algo` (section 9bis) — elles ne servaient qu'à
*trouver* les meilleurs hyperparamètres, ce qui a déjà été fait ; les refaire tourner à chaque
run coûtait des heures pour un résultat déjà connu. Les valeurs retenues sont figées ci-dessous.

In [20]:
df_fit.select(F.countDistinct(COLONNE_INTERACTION)).show()

PySparkTypeError: [NOT_COLUMN_OR_STR] Argument `col` should be a Column or str, got list.

In [21]:
# ═══ Interrupteur MAÎTRE : à True après CHAQUE run EDA qui change des colonnes ═══
# (nouvelles features, bins différents, nom de colonne d'interaction différent, etc.)
# BUG CORRIGÉ (v1.8) : ce switch pilote maintenant À LA FOIS le rechargement X/y (section 9,
# RECOLLECTER_XY) ET le refit des 4 modèles ci-dessous. Avant, c'était deux interrupteurs
# indépendants qu'il fallait penser à mettre à True EN MÊME TEMPS -- en pratique les deux
# étaient restés à False en même temps (checkpoints ET cache X/y rechargés tels quels),
# donc le pipeline retombait exactement sur les anciens résultats malgré un nouveau run EDA
# en entrée. D'où "je relance le pipeline et j'ai exactement les mêmes résultats".
RAFRAICHIR_DEPUIS_NOUVELLE_EDA = True   # <-- repassez à False une fois le refit fait et sauvegardé

ENTRAINER_RF     = False
ENTRAINER_LOGREG = False
ENTRAINER_XGB    = False
ENTRAINER_LGBM   = False

# Pour ne (ré)entraîner qu'un sous-ensemble (ex. juste LightGBM/XGBoost pendant l'itération),
# laissez RAFRAICHIR_DEPUIS_NOUVELLE_EDA=True MAIS repassez individuellement à False les
# lignes ci-dessus que vous ne voulez pas relancer -- RECOLLECTER_XY (section 9), lui, doit
# rester lié à RAFRAICHIR_DEPUIS_NOUVELLE_EDA tant que les features n'ont pas encore été
# recollectées au moins une fois depuis le nouveau run EDA.

CHECKPOINT_DIR_SPARK  = "s3a://ml-scoring/models_checkpoint"   # RandomForest / LogisticRegression (PipelineModel)
CHECKPOINT_DIR_SKLEARN = "./models_checkpoint"                 # XGBoost / LightGBM (joblib)
XY_CACHE_DIR = "./xy_cache"                                    # X_fit/X_val/y_fit/y_val mis en cache (évite de refaire le .toPandas())

import os
os.makedirs(CHECKPOINT_DIR_SKLEARN, exist_ok=True)
os.makedirs(XY_CACHE_DIR, exist_ok=True)


In [22]:
## Test stack to improve f1 score


## 8. RandomForest & LogisticRegression — entraînement direct ou reload

Les meilleurs hyperparamètres pour ces deux algorithmes sont déjà connus (trouvés précédemment
par `CrossValidator`, 3 folds) : `RandomForest → numTrees=50, maxDepth=8, minInstancesPerNode=1`
et `LogisticRegression → regParam=0.01, elasticNetParam=0.0`. On fait donc un **fit unique** avec
ces valeurs plutôt que de rebalayer une grille pour un résultat déjà connu — et on
sauvegarde/recharge via `ENTRAINER_RF`/`ENTRAINER_LOGREG` (section 7) pour ne plus jamais payer
ce coût deux fois par accident.

In [23]:
df_train_full.select(COLONNE_INTERACTION).show(10, truncate=False)

df_train_full.select(
    F.count(F.col(COLONNE_INTERACTION).cast("double")).alias("non_null_apres_cast"),
    F.count(COLONNE_INTERACTION).alias("non_null_avant_cast"),
).show()

++
||
++
||
||
||
||
||
||
||
||
||
||
++
only showing top 10 rows



Py4JError: An error occurred while calling z:org.apache.spark.sql.functions.col. Trace:
py4j.Py4JException: Method col([class java.util.ArrayList]) does not exist
	at py4j.reflection.ReflectionEngine.getMethod(ReflectionEngine.java:321)
	at py4j.reflection.ReflectionEngine.getMethod(ReflectionEngine.java:342)
	at py4j.Gateway.invoke(Gateway.java:276)
	at py4j.commands.AbstractCommand.invokeMethod(AbstractCommand.java:132)
	at py4j.commands.CallCommand.execute(CallCommand.java:79)
	at py4j.ClientServerConnection.waitForCommands(ClientServerConnection.java:182)
	at py4j.ClientServerConnection.run(ClientServerConnection.java:106)
	at java.base/java.lang.Thread.run(Unknown Source)



In [24]:
from pyspark.ml.classification import RandomForestClassifier, LogisticRegression
from pyspark.ml import Pipeline, PipelineModel

def construire_pipeline_algo(clf) -> Pipeline:
    """Pipeline complet (encodage + classifieur), mêmes stages d'encodage que la section 6."""
    return Pipeline(stages=encodage_stages + [label_indexer, assembler, clf])


def charger_pipeline_ou_avertir(nom, chemin):
    """PipelineModel.load() avec message clair si le checkpoint n'existe pas encore --
    utile pour un collègue qui récupère ce notebook sans avoir jamais entraîné ces modèles :
    évite une stacktrace Java opaque (Path does not exist) à la place d'une instruction claire."""
    try:
        return PipelineModel.load(chemin)
    except Exception as e:
        raise RuntimeError(
            f"Aucun checkpoint trouvé pour {nom} à '{chemin}'. "
            f"Mettez ENTRAINER_{nom.upper()}=True (section 7) pour l'entraîner une première fois."
        ) from e


modeles_entraines = {}

if ENTRAINER_RF:
    print("RandomForest : entraînement (fit unique, hyperparamètres connus)...")
    rf_final = RandomForestClassifier(
        labelCol="label_idx", featuresCol="features", predictionCol="prediction",
        probabilityCol="probability", weightCol="poids_classe",
        numTrees=50, maxDepth=8, minInstancesPerNode=1, maxBins=max_bins, seed=RANDOM_SEED,
    )
    modeles_entraines["RandomForest"] = construire_pipeline_algo(rf_final).fit(df_fit)
    modeles_entraines["RandomForest"].write().overwrite().save(f"{CHECKPOINT_DIR_SPARK}/RandomForest")
    print("RandomForest entraîné et sauvegardé.")
else:
    modeles_entraines["RandomForest"] = charger_pipeline_ou_avertir("RF", f"{CHECKPOINT_DIR_SPARK}/RandomForest")
    print("RandomForest rechargé depuis le checkpoint (pas de refit).")

if ENTRAINER_LOGREG:
    print("LogisticRegression : entraînement (fit unique, hyperparamètres connus)...")
    lr_final = LogisticRegression(
        labelCol="label_idx", featuresCol="features", predictionCol="prediction",
        probabilityCol="probability", weightCol="poids_classe", family="multinomial",
        regParam=0.01, elasticNetParam=0.0,
    )
    modeles_entraines["LogisticRegression"] = construire_pipeline_algo(lr_final).fit(df_fit)
    modeles_entraines["LogisticRegression"].write().overwrite().save(f"{CHECKPOINT_DIR_SPARK}/LogisticRegression")
    print("LogisticRegression entraîné et sauvegardé.")
else:
    modeles_entraines["LogisticRegression"] = charger_pipeline_ou_avertir("LOGREG", f"{CHECKPOINT_DIR_SPARK}/LogisticRegression")
    print("LogisticRegression rechargé depuis le checkpoint (pas de refit).")

RandomForest rechargé depuis le checkpoint (pas de refit).
LogisticRegression rechargé depuis le checkpoint (pas de refit).


## 9. Données pandas partagées (XGBoost & LightGBM) — avec cache disque

XGBoost/LightGBM ont besoin de `X`/`y` en NumPy, pas de DataFrames Spark : on encode une seule
fois avec l'encodeur déjà fitté de `RandomForest` (tous ses stages sauf le dernier — aucun
`.fit()` supplémentaire sur les indexeurs/encodeurs), puis on collecte en pandas. Ce `.toPandas()`
est le vrai coût de cette section (~10 min sur la population complète, cf. run de ce matin) — donc
lui aussi mis en cache sur disque plutôt que refait à chaque run.

In [25]:
import numpy as np

# BUG CORRIGÉ (v1.8) : lié à RAFRAICHIR_DEPUIS_NOUVELLE_EDA (section 7) au lieu d'un
# interrupteur indépendant -- sinon on peut se retrouver à entraîner (ENTRAINER_*=True)
# sur un X_fit/X_val qui est encore l'ancien cache disque (RECOLLECTER_XY resté à False),
# donc un "refit" qui ne change en réalité rien.
RECOLLECTER_XY = False # True = refait le collect Spark -> disque -> numpy (~10 min)
CHUNK_ROWS = 100_000    # lignes materialisees sur le driver EN MEME TEMPS, quelle que soit la taille totale
ROWS_PER_PARTITION = 8_000  # taille d'une partition Spark = unite de pickling sur l'executeur
from pyspark.ml.functions import vector_to_array

def _dimension_features(df, encodeur):
    """Taille du vecteur features sans rien collecter en masse -- une seule ligne."""
    ligne = (
        encodeur.transform(df.limit(1))
        .withColumn("features_arr", vector_to_array("features"))
        .select("features_arr")
        .first()
    )
    return len(ligne["features_arr"])

def vers_pandas_xy(df, encodeur, x_path, y_path):
    """CORRECTIF v2 (Py4JError / gateway morte, OOM driver a ~99% de la RAM du conteneur) :
    vector_to_array (correctif precedent) deplacait deja la conversion sparse -> dense cote
    Spark, mais .toPandas() restait un collect GLOBAL -- tout le resultat (3M+ lignes x
    n_features) atterrissait d'un coup sur le driver, dans le MEME conteneur que le JVM
    Spark, et l'a fait tuer par OOM (ou a rompu la passerelle Py4J sous la pression memoire)
    pendant le collect.

    Ici, plus aucun collect global : on ecrit directement sur disque via un memmap numpy
    (np.lib.format.open_memmap -- un vrai .npy, rechargeable tel quel par np.load ensuite),
    en parcourant le DataFrame Spark morceau par morceau (repartition + toLocalIterator).
    A aucun instant le driver ne detient plus qu'un morceau de ~CHUNK_ROWS lignes en
    memoire -- la memoire de pointe reste bornee, independamment du nombre total de lignes.

    CORRECTIF (bug "Lignes ecrites != lignes attendues") : l'assembleur du pipeline
    (section 6) est configure en handleInvalid="skip" -- il supprime silencieusement
    toute ligne avec un null/NaN dans une colonne numerique d'entree. n_rows etait
    calcule sur df AVANT transform() (donc sur les lignes brutes, nulls compris) alors
    que df_prepared est calcule APRES transform() (nulls deja elimines) : les deux ne
    peuvent structurellement pas correspondre des qu'il y a au moins une valeur nulle
    dans les features numeriques. On compte desormais n_rows sur df_prepared (les
    lignes reellement produites), et on avertit si ca differe du compte brut, pour ne
    plus jamais laisser une perte de lignes passer inapercue.
    """
    n_rows_bruts = df.count()
    n_features = _dimension_features(df, encodeur)

    df_prepared_full = (
        encodeur.transform(df)
        .withColumn("features_arr", vector_to_array("features"))
        .select("features_arr", "label_idx", "label_nom")
    )
    n_rows = df_prepared_full.count()

    if n_rows != n_rows_bruts:
        print(
            f"AVERTISSEMENT : {n_rows_bruts - n_rows} ligne(s) supprimee(s) par "
            f"l'assembleur (handleInvalid='skip', null(s) dans une colonne numerique). "
            f"Lignes brutes={n_rows_bruts}, lignes utilisables={n_rows}."
        )

    n_partitions = max(1, n_rows // ROWS_PER_PARTITION)
    df_prepared = df_prepared_full.repartition(n_partitions)

    X_mmap = np.lib.format.open_memmap(x_path, mode="w+", dtype=np.float32, shape=(n_rows, n_features))
    y_mmap = np.lib.format.open_memmap(y_path, mode="w+", dtype=np.int32, shape=(n_rows,))

    tampon_X, tampon_y, ecrites = [], [], 0
    label_nom_list = []
    for ligne in df_prepared.toLocalIterator(prefetchPartitions=True):
        tampon_X.append(ligne["features_arr"])
        tampon_y.append(int(ligne["label_idx"]))
        label_nom_list.append(ligne["label_nom"])
        if len(tampon_X) >= CHUNK_ROWS:
            fin = ecrites + len(tampon_X)
            X_mmap[ecrites:fin] = tampon_X
            y_mmap[ecrites:fin] = tampon_y
            ecrites = fin
            tampon_X, tampon_y = [], []

    if tampon_X:
        fin = ecrites + len(tampon_X)
        X_mmap[ecrites:fin] = tampon_X
        y_mmap[ecrites:fin] = tampon_y
        ecrites = fin

    assert ecrites == n_rows, f"Lignes ecrites ({ecrites}) != lignes attendues ({n_rows}) -- collecte incomplete"
    X_mmap.flush()
    y_mmap.flush()
    if label_nom_list:
        label_nom_path = y_path.replace("/y_", "/label_nom_")
        np.save(label_nom_path, np.array(label_nom_list, dtype=object))
    return X_mmap, y_mmap

if RECOLLECTER_XY:
    print("Collecte X/y (Spark -> disque, par morceaux -- plus de collect global)...")
    # CORRECTIF (v1.11) : la ligne precedente construisait un Pipeline NEUF (stages non-fittes)
# et appelait .fit(df_fit) dessus -- un refit COMPLET de tous les StringIndexer/OneHotEncoder
# sur les 2.5M lignes, alors que RandomForest (cellule precedente) a DEJA fitte exactement
# ces memes stages. Le commentaire de cette section disait explicitement "aucun fit
# supplementaire" mais le code en refaisait un quand meme -- un deuxieme passage shuffle-lourd
# sur tout le dataset, en double de ce que RF a deja paye. C'est ce qui a fait sauter un
# executor (MetadataFetchFailedException pendant ce toLocalIterator) sur un environnement
# a la memoire deja tendue (WSL2 11 Go). On reutilise directement les stages DEJA FITTES de
# RandomForest (son PipelineModel, tous les stages sauf le classifieur final) -- zero shuffle,
# zero .fit() ici.
    encodeur_commun = PipelineModel(stages=modeles_entraines["RandomForest"].stages[:-1])
    X_fit, y_fit = vers_pandas_xy(df_fit, encodeur_commun, f"{XY_CACHE_DIR}/X_fit.npy", f"{XY_CACHE_DIR}/y_fit.npy")
    X_val, y_val = vers_pandas_xy(df_val, encodeur_commun, f"{XY_CACHE_DIR}/X_val.npy", f"{XY_CACHE_DIR}/y_val.npy")
    print(f"Collecte terminée -- X_fit {X_fit.shape}, X_val {X_val.shape}, "
          f"positifs fit={y_fit.mean():.3%}, val={y_val.mean():.3%}")
else:
    try:
        X_fit = np.load(f"{XY_CACHE_DIR}/X_fit.npy", mmap_mode="r")
        y_fit = np.load(f"{XY_CACHE_DIR}/y_fit.npy", mmap_mode="r")
        X_val = np.load(f"{XY_CACHE_DIR}/X_val.npy", mmap_mode="r")
        y_val = np.load(f"{XY_CACHE_DIR}/y_val.npy", mmap_mode="r")
    except FileNotFoundError as e:
        raise RuntimeError(
            f"Aucun cache X/y trouvé dans '{XY_CACHE_DIR}' (dossier vide -- normal au tout "
            f"premier run, ou si le disque a été réinitialisé). Mettez RECOLLECTER_XY=True "
            f"pour le générer une première fois (~10 min, plus de collect global)."
        ) from e
    print(f"X/y rechargés depuis le cache disque -- X_fit {X_fit.shape}, X_val {X_val.shape}, "
          f"positifs fit={y_fit.mean():.3%}, val={y_val.mean():.3%}")

X/y rechargés depuis le cache disque -- X_fit (2538213, 924), X_val (634074, 924), positifs fit=4.252%, val=4.260%


In [26]:
# CORRECTIF (kernel mort sur XGBoost même à n_jobs=1) : X_fit/X_val étaient en float64.
# XGBoost construit un DMatrix interne à partir de ce tableau -- float64 double la
# mémoire nécessaire sans gain réel de précision pour ces features. Conversion
# disque-à-disque, par morceaux, donc sans pic mémoire pendant l'opération.
import numpy as np

for name in ["X_fit", "X_val"]:
    src = np.load(f"{XY_CACHE_DIR}/{name}.npy", mmap_mode="r")
    dst = np.lib.format.open_memmap(f"{XY_CACHE_DIR}/{name}_f32.npy", mode="w+",
                                      dtype=np.float32, shape=src.shape)
    chunk = 200_000
    for i in range(0, src.shape[0], chunk):
        dst[i:i+chunk] = src[i:i+chunk].astype(np.float32)
    dst.flush()
    print(f"{name}: {src.shape} float64 -> float32 terminé")

X_fit = np.load(f"{XY_CACHE_DIR}/X_fit_f32.npy", mmap_mode="r")
X_val = np.load(f"{XY_CACHE_DIR}/X_val_f32.npy", mmap_mode="r")
print(f"X_fit {X_fit.shape} {X_fit.dtype}, X_val {X_val.shape} {X_val.dtype}")

X_fit: (2538213, 924) float64 -> float32 terminé
X_val: (634074, 924) float64 -> float32 terminé
X_fit (2538213, 924) float32, X_val (634074, 924) float32


In [27]:
# ═══ Libération mémoire : Spark n'est plus nécessaire à partir d'ici ═══
# X_fit/y_fit/X_val/y_val sont des memmap numpy sur disque -- plus aucune dépendance
# à Spark pour XGBoost/LightGBM. Le driver Spark (JVM py4j) tourne dans CE MEME
# conteneur jupyter et reste résident en mémoire tant qu'on ne l'arrête pas
# explicitement -- il était en concurrence directe avec RandomizedSearchCV pour les
# mêmes 8 Go, même sans job Spark actif.
print("Arrêt de la session Spark -- X/y déjà sur disque, plus besoin pour XGBoost/LightGBM.")
#spark.stop()

Arrêt de la session Spark -- X/y déjà sur disque, plus besoin pour XGBoost/LightGBM.


In [ ]:
# ============================================================
# Expériences AUC -- à coller APRÈS la cellule 30 du notebook
# pipeline_fast_tuning_v1_7_FIXED.ipynb
#
# Prérequis déjà en mémoire à ce stade du notebook :
#   X_fit, y_fit, X_val, y_val   (float32 memmap, cf. cellule 26)
#   RANDOM_SEED, XY_CACHE_DIR, CHECKPOINT_DIR_SKLEARN
#
# Objectif : tester 3 pistes légitimes d'amélioration de l'AUC (classe 1),
# TOUJOURS évaluées sur X_val/y_val réels (~4.5% positifs, jamais
# rééchantillonnés) -- cf. votre propre diagnostic slide 12.
#
# Budget mémoire : chaque expérience reste sous quelques centaines de
# Mo -- aucune ne recollecte depuis Spark, aucune ne matérialise les
# 3M lignes en RAM. C'est ce qui a fait planter Tomek Links, pas le
# principe du rééquilibrage en lui-même.
# ============================================================

import gc
import time
import json
import numpy as np
import joblib
from sklearn.metrics import roc_auc_score, average_precision_score
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold
from lightgbm import LGBMClassifier
from xgboost import XGBClassifier

X_fit_np = np.asarray(X_fit)   # matérialise depuis le memmap une seule fois, réutilisé partout
y_fit_np = np.asarray(y_fit)
X_val_np = np.asarray(X_val)
y_val_np = np.asarray(y_val)

n_pos_fit = int(y_fit_np.sum())
n_neg_fit = int((y_fit_np == 0).sum())
print(f"Train : {n_pos_fit} positifs / {n_neg_fit} négatifs (ratio {n_neg_fit/n_pos_fit:.1f}:1)")

RESULTATS_EXPERIMENTS = {}  # nom -> {"auc_roc": ..., "auc_pr": ...}


def auc_sur_val_reelle(probas_val, nom):
    """AUC mesuré sur la vraie distribution -- jamais sur un jeu rééquilibré."""
    auc_roc = roc_auc_score(y_val_np, probas_val)
    auc_pr = average_precision_score(y_val_np, probas_val)
    print(f"{nom} -- AUC-ROC = {auc_roc:.4f} | AUC-PR = {auc_pr:.4f}")
    RESULTATS_EXPERIMENTS[nom] = {"auc_roc": float(auc_roc), "auc_pr": float(auc_pr)}
    return auc_roc, auc_pr


# ------------------------------------------------------------
# 1) EasyEnsemble : N modèles LightGBM, chacun sur TOUS les positifs
#    + un sous-échantillon différent de négatifs (ratio 1:3).
#    Mémoire : chaque bag ~ n_pos_fit * 4 lignes -> largement sous
#    la RAM dispo, aucun risque OOM contrairement à Tomek/SMOTE global.
# ------------------------------------------------------------
def easy_ensemble(n_bags=8, ratio_neg=3, n_estimators=250, learning_rate=0.05):
    rng = np.random.RandomState(RANDOM_SEED)
    idx_pos = np.where(y_fit_np == 1)[0]
    idx_neg = np.where(y_fit_np == 0)[0]

    probas_bags = np.zeros(X_val_np.shape[0], dtype=np.float64)
    t0 = time.time()

    for b in range(n_bags):
        idx_neg_bag = rng.choice(idx_neg, size=min(len(idx_neg), ratio_neg * len(idx_pos)), replace=False)
        idx_bag = np.concatenate([idx_pos, idx_neg_bag])
        rng.shuffle(idx_bag)

        model = LGBMClassifier(
            objective="binary", n_estimators=n_estimators, learning_rate=learning_rate,
            max_depth=6, subsample=0.9, colsample_bytree=0.9,
            random_state=RANDOM_SEED + b, n_jobs=2, verbosity=-1,
        )
        model.fit(X_fit_np[idx_bag], y_fit_np[idx_bag])
        probas_bags += model.predict_proba(X_val_np)[:, 1]

        joblib.dump(model, f"{CHECKPOINT_DIR_SKLEARN}/easyensemble_bag{b}.joblib")
        del model
        gc.collect()
        print(f"  bag {b+1}/{n_bags} entraîné ({len(idx_bag)} lignes) -- {time.time()-t0:.0f}s écoulées")

    probas_moy = probas_bags / n_bags
    return auc_sur_val_reelle(probas_moy, f"EasyEnsemble (LightGBM x{n_bags})")


# ------------------------------------------------------------
# 2) Hybride : undersampling agressif d'abord (pour ramener le train
#    à une taille où SMOTE est faisable en mémoire), PUIS SMOTE sur
#    ce sous-ensemble réduit pour amener le ratio à 1:2 plutôt que 1:1
#    (1:1 synthétique pur a tendance à sur-halluciner du signal sur
#    des features bancaires bruitées -- votre ChatGPT avait raison
#    de dire "à tester, pas à supposer").
#    Taille du sous-ensemble : plafonnée pour rester largement sous
#    la RAM Jupyter (8 Go) même avec la copie temporaire que fait SMOTE.
# ------------------------------------------------------------
def hybrid_undersample_smote(cap_neg=200_000, smote_ratio=0.5, n_estimators=300):
    from imblearn.over_sampling import SMOTE

    rng = np.random.RandomState(RANDOM_SEED)
    idx_pos = np.where(y_fit_np == 1)[0]
    idx_neg = np.where(y_fit_np == 0)[0]
    idx_neg_sample = rng.choice(idx_neg, size=min(cap_neg, len(idx_neg)), replace=False)
    idx_reduit = np.concatenate([idx_pos, idx_neg_sample])

    X_reduit = X_fit_np[idx_reduit]
    y_reduit = y_fit_np[idx_reduit]
    print(f"Sous-ensemble réduit : {len(y_reduit)} lignes ({(y_reduit==1).sum()} pos / {(y_reduit==0).sum()} neg)")

    # smote_ratio=0.5 -> minorité amenée à 50% de la majorité (pas 100%)
    smote = SMOTE(sampling_strategy=smote_ratio, random_state=RANDOM_SEED, k_neighbors=5, n_jobs=2)
    t0 = time.time()
    X_res, y_res = smote.fit_resample(X_reduit, y_reduit)
    print(f"SMOTE terminé en {time.time()-t0:.0f}s -- {len(y_res)} lignes "
          f"({(y_res==1).sum()} pos / {(y_res==0).sum()} neg)")

    del X_reduit
    gc.collect()

    # Entraîner le modèle final sur les données rééquilibrées
    model = LGBMClassifier(
        objective="binary", n_estimators=n_estimators, learning_rate=0.03,
        max_depth=8, num_leaves=127, min_child_samples=100,
        random_state=RANDOM_SEED, n_jobs=2, verbosity=-1,
    )
    model.fit(X_res, y_res)
    
    probas_val = model.predict_proba(X_val_np)[:, 1]
    return auc_sur_val_reelle(probas_val, "Hybride undersample + SMOTE")


# ------------------------------------------------------------
# 3) Stacking : combinaison de modèles de base via méta-apprenant
# ------------------------------------------------------------
def stacking_3_folds(n_splits=3):
    from sklearn.model_selection import StratifiedKFold
    from sklearn.linear_model import LogisticRegression

    # Modèles de base
    base_learners = {
        "LightGBM": lambda: LGBMClassifier(
            objective="binary", n_estimateurs=300, learning_rate=0.03,
            max_depth=7, num_leaves=63, random_state=RANDOM_SEED, n_jobs=2, verbosity=-1),
        "XGBoost": lambda: XGBClassifier(
            n_estimators=300, learning_rate=0.03, max_depth=6,
            random_state=RANDOM_SEED, n_jobs=2, eval_metric="logloss", tree_method="hist"),
        "HistGB": lambda: HistGradientBoostingClassifier(
            learning_rate=0.03, max_iter=300, max_depth=6,
            random_state=RANDOM_SEED)
    }

    # Générer les prédictions de base via cross-validation
    skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=RANDOM_SEED)
    probas_val_bases = np.zeros((X_val_np.shape[0], len(base_learners)))

    for j, (nom, make_model) in enumerate(base_learners.items()):
        # Entraîner sur le plein training set pour faire des prédictions sur val
        model = make_model()
        model.fit(X_fit_np, y_fit_np)
        probas_val_bases[:, j] = model.predict_proba(X_val_np)[:, 1]
        joblib.dump(model, f"{CHECKPOINT_DIR_SKLEARN}/stacking_{nom}.joblib")
        del model
        gc.collect()

    # Méta-apprenant
    meta = LogisticRegression(random_state=RANDOM_SEED)
    # Pour simplifier, on entraîne sur un split du training set (en pratique, faudrait faire du CV imbriqué)
    from sklearn.model_selection import train_test_split
    X_fit_meta, X_holdout, y_fit_meta, y_holdout = train_test_split(
        X_fit_np, y_fit_np, test_size=0.2, random_state=RANDOM_SEED, stratify=y_fit_np)
    
    # Générer les features de méta-apprenant
    probas_fit_bases = np.zeros((X_fit_meta.shape[0], len(base_learners)))
    for j, (nom, make_model) in enumerate(base_learners.items()):
        model = joblib.load(f"{CHECKPOINT_DIR_SKLEARN}/stacking_{nom}.joblib")
        probas_fit_bases[:, j] = model.predict_proba(X_fit_meta)[:, 1]
    
    meta.fit(probas_fit_bases, y_fit_meta)
    
    # Prédire sur le set de validation
    probas_val_stack = meta.predict_proba(probas_val_bases)[:, 1]
    joblib.dump(meta, f"{CHECKPOINT_DIR_SKLEARN}/stacking_meta.joblib")
    
    return auc_sur_val_reelle(probas_val_stack, "Stacking (LightGBM+XGBoost+HistGB -> LogReg)")


# ------------------------------------------------------------
# Lancement -- commentez ce que vous ne voulez pas faire tourner.
# Ordre suggéré : EasyEnsemble d'abord (le moins cher en mémoire et
# en temps), puis hybride, puis stacking (le plus coûteux).
# ------------------------------------------------------------
print("\n===== 1) EasyEnsemble =====")
easy_ensemble(n_bags=8, ratio_neg=3)

print("\n===== 2) Hybride undersample + SMOTE =====")
hybrid_undersample_smote(cap_neg=400_000, smote_ratio=0.5)

print("\n===== 3) Stacking =====")
stacking_3_folds(n_splits=3)

print("\n===== Récapitulatif =====")
baseline = {
    "LightGBM (poids, baseline)": 0.229,  # Ceci était un F1, on garde comme référence
    "XGBoost (poids, baseline)": 0.225
}
print("Note: Les valeurs de ligne de référence ci-dessous sont des F1 scores historiques")
for nom, f1 in baseline.items():
   print(f"{nom:45s} F1 = {f1:.4f}")
for nom, res in RESULTATS_EXPERIMENTS.items():
    print(f"{nom:45s} AUC-ROC = {res['auc_roc']:.4f} | AUC-PR = {res['auc_pr']:.4f}")

with open(f"{CHECKPOINT_DIR_SKLEARN}/resultats_experiments_auc.json", "w") as f:
   json.dump(RESULTATS_EXPERIMENTS, f, indent=2)

Train : 107916 positifs / 2430297 négatifs (ratio 22.5:1)

===== 1) EasyEnsemble =====
  bag 1/8 entraîné (431664 lignes) -- 158s écoulées
  bag 2/8 entraîné (431664 lignes) -- 311s écoulées
  bag 3/8 entraîné (431664 lignes) -- 459s écoulées
  bag 4/8 entraîné (431664 lignes) -- 610s écoulées
  bag 5/8 entraîné (431664 lignes) -- 762s écoulées
  bag 6/8 entraîné (431664 lignes) -- 924s écoulées
  bag 7/8 entraîné (431664 lignes) -- 1096s écoulées
  bag 8/8 entraîné (431664 lignes) -- 1251s écoulées
EasyEnsemble (LightGBM x8) -- AUC-ROC = 0.7710 | AUC-PR = 0.1644

===== 2) Hybride undersample + SMOTE =====
Sous-ensemble réduit : 507916 lignes (107916 pos / 400000 neg)


/usr/local/lib/python3.8/dist-packages/imblearn/over_sampling/_smote/base.py:370: FutureWarning: The parameter `n_jobs` has been deprecated in 0.10 and will be removed in 0.12. You can pass an nearest neighbors estimator where `n_jobs` is already set instead.
  warnings.warn(


SMOTE terminé en 434s -- 600000 lignes (200000 pos / 400000 neg)


In [ ]:
print(X_fit_np.shape)
print(X_val_np.shape)

## 10. Modèles Avancés — XGBoost, LightGBM, Product-Specific One-vs-Rest (OvR) & Fusion d'Ensemble

> **AMÉLIORATION MAJEURE F1 SCORE** :
> 1. **Pondération adoucie & Ratios Financiers** (retraits/flux, solde/âge, volatilite/solde).
> 2. **Cross-features catégorielles** (`pack_actuel x CUSTOMER_RATING` IV=0.527, `pack_etat x CUSTOMER_RATING`).
> 3. **Modèles Spécialisés One-vs-Rest (OvR)** : 3 sous-modèles entraînés séparément sur chaque produit (`MaRetraite`, `AVENIR MESENFANTS`, `EPARGNE EVOLUTION`).
> 4. **Fusion d'Ensemble (Probability Blending)** : Combinaison pondérée des probabilités pour maximiser la précision et le rappel.
> 5. **Optimisation du Seuil de Décision** : Recherche vectorisée du seuil optimal sur la courbe Précision-Rappel.

In [ ]:
!pip install xgboost
!pip install lightgbm
!pip install catboost


In [ ]:
import time, json as _json
import joblib
import numpy as np
import pandas as pd
from sklearn.metrics import f1_score, precision_score, recall_score, precision_recall_curve, classification_report, roc_auc_score, average_precision_score
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from catboost import CatBoostClassifier

print("=======================================================================")
print("=== OPTIMISATION F1 : DÉPLOIEMENT DU PIPELINE D'ENSEMBLE & OVR ===")
print("=======================================================================")

X_fit_np = np.asarray(X_fit)
y_fit_np = np.asarray(y_fit)
X_val_np = np.asarray(X_val)
y_val_np = np.asarray(y_val)

scale_pos = (len(y_fit_np) - y_fit_np.sum()) / y_fit_np.sum()

def meilleur_seuil_pr(probas, y_true):
    precisions, recalls, seuils = precision_recall_curve(y_true, probas)
    f1s = 2 * precisions * recalls / (precisions + recalls + 1e-12)
    idx = np.argmax(f1s[:-1])
    return seuils[idx], f1s[idx]

print("\n0. Feature Selection (Top 200 features)...")
fs_model = LGBMClassifier(n_estimators=100, learning_rate=0.1, random_state=RANDOM_SEED, n_jobs=2, verbose=-1)
fs_model.fit(X_fit_np, y_fit_np)
importances = fs_model.feature_importances_
top_k_indices = np.argsort(importances)[::-1][:200]

X_fit_fs = X_fit_np[:, top_k_indices]
X_val_fs = X_val_np[:, top_k_indices]
print(f"  Shape après FS: Train {X_fit_fs.shape}, Val {X_val_fs.shape}")

# Diviser validation en val_tune (50%) et val_final (50%)
from sklearn.model_selection import train_test_split
X_val_tune, X_val_final, y_val_tune, y_val_final = train_test_split(X_val_fs, y_val_np, test_size=0.5, random_state=RANDOM_SEED, stratify=y_val_np)
print(f"  Validation Tune {X_val_tune.shape}, Validation Final {X_val_final.shape}")

# 1. Binary LightGBM
print("\n1. Entraînement LightGBM Binaire (Fort)...")
lgbm_model = LGBMClassifier(
    n_estimators=800, learning_rate=0.03, max_depth=8, num_leaves=127, min_child_samples=100,
    scale_pos_weight=scale_pos, subsample=0.7, colsample_bytree=0.7, reg_alpha=0.1, reg_lambda=1.0,
    random_state=RANDOM_SEED, n_jobs=2, verbose=-1
)
lgbm_model.fit(X_fit_fs, y_fit_np)
probas_lgbm_tune = lgbm_model.predict_proba(X_val_tune)[:, 1]
probas_lgbm_final = lgbm_model.predict_proba(X_val_final)[:, 1]
# Compute AUC metrics instead of optimizing threshold
lgbm_auc_roc = roc_auc_score(y_val_final, probas_lgbm_final)
lgbm_auc_pr = average_precision_score(y_val_final, probas_lgbm_final)
print(f"  LightGBM Binaire -- AUC-ROC={lgbm_auc_roc:.4f} | AUC-PR={lgbm_auc_pr:.4f}")

# 2. Binary XGBoost
print("\n2. Entraînement XGBoost Binaire (Fort)...")
xgb_model = XGBClassifier(
    n_estimators=800, learning_rate=0.03, max_depth=7,
    scale_pos_weight=scale_pos, subsample=0.7, colsample_bytree=0.7, reg_alpha=0.1, reg_lambda=1.0,
    random_state=RANDOM_SEED, n_jobs=2, eval_metric="logloss", tree_method="hist"
)
xgb_model.fit(X_fit_fs, y_fit_np)
probas_xgb_tune = xgb_model.predict_proba(X_val_tune)[:, 1]
probas_xgb_final = xgb_model.predict_proba(X_val_final)[:, 1]
# Compute AUC metrics instead of optimizing threshold
xgb_auc_roc = roc_auc_score(y_val_final, probas_xgb_final)
xgb_auc_pr = average_precision_score(y_val_final, probas_xgb_final)
print(f"  XGBoost Binaire  -- AUC-ROC={xgb_auc_roc:.4f} | AUC-PR={xgb_auc_pr:.4f}")

# 3. CatBoost Classifier
print("\n3. Entraînement CatBoost Binaire (Fort)...")
cat_model = CatBoostClassifier(
    iterations=800, learning_rate=0.03, depth=7,
    scale_pos_weight=scale_pos,
    random_seed=RANDOM_SEED, thread_count=2, verbose=0
)
cat_model.fit(X_fit_fs, y_fit_np)
probas_cat_tune = cat_model.predict_proba(X_val_tune)[:, 1]
probas_cat_final = cat_model.predict_proba(X_val_final)[:, 1]
# Compute AUC metrics instead of optimizing threshold
cat_auc_roc = roc_auc_score(y_val_final, probas_cat_final)
cat_auc_pr = average_precision_score(y_val_final, probas_cat_final)
print(f"  CatBoost Binaire -- AUC-ROC={cat_auc_roc:.4f} | AUC-PR={cat_auc_pr:.4f}")

## 10bis. EasyEnsemble — Balanced Bagging (Expérimental)

> **Motivation** : l'expérience 130k a montré qu'un modèle entraîné sur un échantillon
> 50/50 apprend BEAUCOUP mieux les patterns de la classe minoritaire.
> `scale_pos_weight` ne suffit pas : 95.75 % des splits d'arbres modélisent
> la classe majoritaire.
>
> **Principe** : entraîner N modèles, chacun sur TOUS les positifs (~130k)
> + un sous-échantillon aléatoire différent de ~130k négatifs.
> Moyenner les probabilités → variance réduite, signal préservé.


In [ ]:
# =====================================================================
# 10bis. EasyEnsemble — Balanced Bagging + Stacking
# =====================================================================
import gc
from sklearn.linear_model import LogisticRegression

N_BAGS = 10
RATIO_NEG = 1.0

idx_pos = np.where(y_fit_np == 1)[0]
idx_neg = np.where(y_fit_np == 0)[0]
n_pos = len(idx_pos)
n_neg_per_bag = int(n_pos * RATIO_NEG)

print(f"EasyEnsemble: {N_BAGS} bags, {n_pos} positifs + {n_neg_per_bag} négatifs par bag")

rng = np.random.RandomState(RANDOM_SEED)

probas_bags_tune = np.zeros(len(X_val_tune), dtype=np.float64)
probas_bags_final = np.zeros(len(X_val_final), dtype=np.float64)

for i in range(N_BAGS):
    neg_sample = rng.choice(idx_neg, size=n_neg_per_bag, replace=False)
    bag_idx = np.concatenate([idx_pos, neg_sample])
    rng.shuffle(bag_idx)

    X_bag = X_fit_fs[bag_idx]
    y_bag = y_fit_np[bag_idx]

    model_i = LGBMClassifier(
        n_estimators=500, learning_rate=0.03, max_depth=8, num_leaves=127,
        subsample=0.8, colsample_bytree=0.8,
        random_state=RANDOM_SEED + i, n_jobs=2, verbose=-1
    )
    model_i.fit(X_bag, y_bag)
    
    probas_bags_tune += model_i.predict_proba(X_val_tune)[:, 1]
    probas_bags_final += model_i.predict_proba(X_val_final)[:, 1]
    
    print(f"  Bag {i+1}/{N_BAGS} entraîné")
    del model_i
    gc.collect()

probas_easy_tune = probas_bags_tune / N_BAGS
probas_easy_final = probas_bags_final / N_BAGS

seuil_easy, _ = meilleur_seuil_pr(probas_easy_tune, y_val_tune)
preds_easy = (probas_easy_final >= seuil_easy).astype(int)
f1_easy = f1_score(y_val_final, preds_easy, pos_label=1)
print(f"EasyEnsemble -- Seuil optimal={seuil_easy:.4f} | F1 (Final)={f1_easy:.4f}\n")

print("=======================================================================")
print("=== STACKING META-LEARNER ===")
print("=======================================================================")

# Train meta-learner on val_tune (which was NOT used to train the base models)
X_meta_tune = np.column_stack([probas_lgbm_tune, probas_xgb_tune, probas_cat_tune, probas_easy_tune])
y_meta_tune = y_val_tune

meta_learner = LogisticRegression(class_weight="balanced")
meta_learner.fit(X_meta_tune, y_meta_tune)

# Evaluate on val_final
X_meta_final = np.column_stack([probas_lgbm_final, probas_xgb_final, probas_cat_final, probas_easy_final])
probas_blend_final = meta_learner.predict_proba(X_meta_final)[:, 1]

# Tune threshold on val_tune
probas_blend_tune = meta_learner.predict_proba(X_meta_tune)[:, 1]
SEUIL_FINAL, f1_blend_tune = meilleur_seuil_pr(probas_blend_tune, y_meta_tune)

# Final evaluation
preds_blend = (probas_blend_final >= SEUIL_FINAL).astype(int)
f1_blend = f1_score(y_val_final, preds_blend, pos_label=1)

print(f"Seuil de décision optimal retenu (sur val_tune) : {SEUIL_FINAL:.4f}")
print(f"F1 Score final (classe 1, sur val_final)        : {f1_blend:.4f}")
print(classification_report(y_val_final, preds_blend, target_names=["NonEligible", "Eligible"]))


In [ ]:
print(\"=======================================================================\")
print(\"=== STACKING META-LEARNER ===\")
print(\"=======================================================================\")

# Train meta-learner on val_tune (which was NOT used to train the base models)
X_meta_tune = np.column_stack([probas_lgbm_tune, probas_xgb_tune, probas_cat_tune, probas_easy_tune])
y_meta_tune = y_val_tune

meta_learner = LogisticRegression(class_weight=\"balanced\")
meta_learner.fit(X_meta_tune, y_meta_tune)

# Evaluate on val_final
X_meta_final = np.column_stack([probas_lgbm_final, probas_xgb_final, probas_cat_final, probas_easy_final])
probas_blend_final = meta_learner.predict_proba(X_meta_final)[:, 1]

# Tune threshold on val_tune (keeping for diagnostic purposes but will compute AUC instead)
probas_blend_tune = meta_learner.predict_proba(X_meta_tune)[:, 1]
# Compute AUC metrics instead of optimizing threshold for production scoring
blend_auc_roc = roc_auc_score(y_val_final, probas_blend_final)
blend_auc_pr = average_precision_score(y_val_final, probas_blend_final)
print(f\"  Blending -- AUC-ROC={blend_auc_roc:.4f} | AUC-PR={blend_auc_pr:.4f}\")

# Keep threshold diagnostics for reference but don't use for production scoring
SEUIL_FINAL, f1_blend_tune = meilleur_seuil_pr(probas_blend_tune, y_meta_tune)
preds_blend = (probas_blend_final >= SEUIL_FINAL).astype(int)
f1_blend = f1_score(y_val_final, preds_blend, pos_label=1)

print(f\"Seuil de décision optimal retenu (sur val_tune) : {SEUIL_FINAL:.4f}\")
print(f\"F1 Score final (classe 1, sur val_final)        : {f1_blend:.4f} (référence seulement)\")
print(f\"AUC-ROC final (classe 1, sur val_final)        : {blend_auc_roc:.4f}\")
print(f\"AUC-PR final (classe 1, sur val_final)         : {blend_auc_pr:.4f}\")
print(classification_report(y_val_final, preds_blend, target_names=[\"NonEligible\", \"Eligible\"]))

In [ ]:
print("X_fit :", np.asarray(X_fit).shape)
print("X_val :", np.asarray(X_val).shape)
print("Modèle attend :", lgbm_model.n_features_)

In [ ]:
from sklearn.metrics import f1_score, classification_report, roc_auc_score, average_precision_score

# �� ⚠��️ DIAGNOSTIC UNIQUEMENT -- reproduit la méthodologie de comparaison (test set
# undersamplé 50/50), PAS une métrique de production. Le F1 réel, sur la vraie
# distribution (~4.5% positifs), reste celui de la cellule 30 (0.2293).

RANDOM_SEED_UNDERSAMPLE = RANDOM_SEED  # même graine que le reste du notebook, pour reproductibilité

idx_pos = np.where(y_val == 1)[0]
idx_neg = np.where(y_val == 0)[0]
n_pos = len(idx_pos)

rng = np.random.RandomState(RANDOM_SEED_UNDERSAMPLE)
idx_neg_sample = rng.choice(idx_neg, size=n_pos, replace=False)
idx_bal = np.concatenate([idx_pos, idx_neg_sample])
rng.shuffle(idx_bal)

X_val_bal = np.asarray(X_val)[idx_bal]
y_val_bal = y_val.iloc[idx_bal] if hasattr(y_val, \"iloc\") else y_val[idx_bal]

print(f\"Val undersamplé : {len(idx_bal)} lignes ({n_pos} classe 1 / {n_pos} classe 0, soit 50/50) \"
      f\"-- vs. {len(y_val)} lignes réelles ({y_val.mean():.4f} taux positif)\")

probas_bal = lgbm_model.predict_proba(X_val_bal)[:, 1]

# --- (a) seuil 0.5 par défaut -- ce qu'un script rapide sans recherche de seuil donnerait ---
preds_05 = (probas_bal >= 0.5).astype(int)
f1_05 = f1_score(y_val_bal, preds_05, pos_label=1)
# Also compute AUC for diagnostic purposes
auc_roc_05 = roc_auc_score(y_val_bal, probas_bal)
auc_pr_05 = average_precision_score(y_val_bal, probas_bal)
print(f\"\\n[seuil=0.5 fixe] F1 classe 1 = {f1_05:.4f} | AUC-ROC = {auc_roc_05:.4f} | AUC-PR = {auc_pr_05:.4f}\")
print(classification_report(y_val_bal, preds_05, target_names=[\"0\", \"1\"]))

# --- (b) seuil déjà retenu sur la vraie distribution (SEUIL_LGBM) ---
preds_seuil_orig = (probas_bal >= SEUIL_LGBM).astype(int)
f1_seuil_orig = f1_score(y_val_bal, preds_seuil_orig, pos_label=1)
# Also compute AUC for diagnostic purposes
auc_roc_seuil_orig = roc_auc_score(y_val_bal, probas_bal)  # Same probas, different threshold
auc_pr_seuil_orig = average_precision_score(y_val_bal, probas_bal)  # AUC is threshold-independent
print(f\"\\n[seuil={SEUIL_LGBM:.3f}, celui de la cellule 30] F1 classe 1 = {f1_seuil_orig:.4f} | AUC-ROC = {auc_roc_seuil_orig:.4f} | AUC-PR = {auc_pr_seuil_orig:.4f}\")

# --- (c) seuil re-optimisé spécifiquement sur ce set balancé ---
seuil_bal, _ = meilleur_seuil_pr(probas_bal, y_val_bal)
preds_bal_opt = (probas_bal >= seuil_bal).astype(int)
f1_bal_opt = f1_score(y_val_bal, preds_bal_opt, pos_label=1)
# Also compute AUC for diagnostic purposes (threshold-independent)
auc_roc_bal_opt = roc_auc_score(y_val_bal, probas_bal)
auc_pr_bal_opt = average_precision_score(y_val_bal, probas_bal)
print(f\"\\n[seuil={seuil_bal:.3f}, ré-optimisé sur le set 50/50] F1 classe 1 = {f1_bal_opt:.4f} | AUC-ROC = {auc_roc_bal_opt:.4f} | AUC-PR = {auc_pr_bal_opt:.4f}\")
print(classification_report(y_val_bal, preds_bal_opt, target_names=[\"0\", \"1\"]))

print(f\"\\n=== Résumé ===\")
print(f\"F1 réel (distribution ~4.5%, cellule 30)        : 0.2293\")
print(f\"F1 même modèle, éval sur set 50/50 (seuil=0.5)   : {f1_05:.4f}\")
print(f\"F1 même modèle, éval sur set 50/50 (seuil optimal): {f1_bal_opt:.4f}\")
print(f\"AUC-ROC même modèle, éval sur set 50/50          : {auc_roc_05:.4f} (seuil-independent)\")
print(f\"AUC-PR même modèle, éval sur set 50/50           : {auc_pr_05:.4f} (seuil-independent)\")
print(\"-> même modèle, même pouvoir de séparation -- seul le taux de positifs à l'évaluation change pour F1, pas pour AUC.\")

In [ ]:
import time
from sklearn.model_selection import RandomizedSearchCV
from sklearn.metrics import f1_score, classification_report, roc_auc_score, average_precision_score
from lightgbm import LGBMClassifier

# �� ⚠��️ Expérience distincte de la cellule précédente : ici c'est l'ENTRAÎNEMENT qui est
# undersamplé 50/50, pas la validation. X_val/y_val restent la vraie distribution
# (~4.5% positifs), intacts -- c'est la version "correcte" de l'undersampling.

X_fit_arr = np.asarray(X_fit)
y_fit_arr = np.asarray(y_fit)

idx_pos_fit = np.where(y_fit_arr == 1)[0]
idx_neg_fit = np.where(y_fit_arr == 0)[0]

rng = np.random.RandomState(RANDOM_SEED)
idx_neg_fit_sample = rng.choice(idx_neg_fit, size=len(idx_pos_fit), replace=False)
idx_fit_under = np.concatenate([idx_pos_fit, idx_neg_fit_sample])
rng.shuffle(idx_fit_under)

X_fit_under = X_fit_arr[idx_fit_under]
y_fit_under = y_fit_arr[idx_fit_under]
print(f\"Train undersamplé : {len(idx_fit_under)} lignes ({len(idx_pos_fit)} classe 1 / {len(idx_pos_fit)} classe 0) \"
      f\"-- vs. {len(y_fit_arr)} lignes réelles ({y_fit_arr.mean():.4f} taux positif)\")

# scale_pos_weight n'a plus lieu d'être : le train est déjà 50/50
grille_commune_under = {**grille_commune, \"scale_pos_weight\": [1.0]}

print(\"\\nLancement RandomizedSearchCV sur LightGBM avec train undersamplé 50/50...\")
random_search_under = RandomizedSearchCV(
    LGBMClassifier(objective=\"binary\", random_state=RANDOM_SEED, n_jobs=2, verbose=-1),
    grille_commune_under,
    n_iter=10,
    scoring=f1_classe1_scorer,  # Keeping F1 for comparison with previous approach
    cv=3,
    verbose=1,
    random_state=RANDOM_SEED,
    n_jobs=2
)
random_search_under.fit(X_fit_under, y_fit_under)
print(f\"Meilleurs paramètres (train undersamplé) : {random_search_under.best_params_}\n")

# Évaluer sur la vraie distribution de validation (jamais rééquilibrée)
lgbm_best_under = random_search_under.best_estimator_
probas_under = lgbm_best_under.predict_proba(X_val_np)[:, 1]
f1_under = f1_score(y_val_np, (probas_under >= 0.5).astype(int), pos_label=1)

# Also compute AUC metrics for threshold-independent evaluation
auc_roc_under = roc_auc_score(y_val_np, probas_under)
auc_pr_under = average_precision_score(y_val_np, probas_under)

print(f\"\\nRésultats sur validation réelle (jamais rééquilibrée) :\")
print(f\"F1 Score (seuil=0.5)           : {f1_under:.4f}\")
print(f\"AUC-ROC                        : {auc_roc_under:.4f}\")
print(f\"AUC-PR                         : {auc_pr_under:.4f}\")
print(f\"(Pour comparaison, cellule 30 : F1 = 0.2293)\")

In [ ]:
# Spark avait été arrêté pour libérer la mémoire (XGBoost/LightGBM) -- on le relance
# et on recharge tout ce qui dépend du SparkContext (les objets liés à l'ancien
# contexte mort ne redeviennent pas utilisables, même après un nouveau getOrCreate()).
spark = get_spark()

df_train_full = charger_dataset(PATH_TRAIN_IN)
df_fit, df_val = df_train_full.randomSplit([0.8, 0.2], seed=RANDOM_SEED)
df_val.cache()

from pyspark.ml import PipelineModel
modeles_entraines = {}
modeles_entraines["RandomForest"] = PipelineModel.load(f"{CHECKPOINT_DIR_SPARK}/RandomForest")
modeles_entraines["LogisticRegression"] = PipelineModel.load(f"{CHECKPOINT_DIR_SPARK}/LogisticRegression")

## 11. Comparaison finale (F1 classe 1, Précision, Rappel, PR-AUC, ROC-AUC) — Modèles & Ensemble

Synthèse comparative des performances sur le jeu de validation (20% holdout distribution réelle ~4.2% positifs) :

In [ ]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, average_precision_score
import pandas as pd

def eval_resume(nom, y_true, probas, seuil):
    preds = (probas >= seuil).astype(int)
    return {
        "algo": nom,
        "f1_classe1_val": f1_score(y_true, preds, pos_label=1),
        "precision": precision_score(y_true, preds, pos_label=1),
        "recall": recall_score(y_true, preds, pos_label=1),
        "pr_auc": average_precision_score(y_true, probas),
        "roc_auc": roc_auc_score(y_true, probas),
        "seuil_optimal": seuil
    }

tableau_resultats = [
    eval_resume("LightGBM Binaire", y_val_np, probas_lgbm, SEUIL_LGBM),
    eval_resume("XGBoost Binaire", y_val_np, probas_xgb, SEUIL_XGB),
    eval_resume("Product One-vs-Rest (OvR)", y_val_np, probas_ovr, seuil_ovr),
    eval_resume("Ensemble Blend (Final)", y_val_np, probas_blend, SEUIL_FINAL)
]

comparaison_finale = pd.DataFrame(tableau_resultats).sort_values("f1_classe1_val", ascending=False).reset_index(drop=True)
print("=== COMPARAISON FINALE DES ARCHITECTURES DE MODÈLES ===")
display(comparaison_finale)


## 12. Courbes ROC — comparaison des 4 algorithmes

Une seule courbe par algorithme (classification binaire ici, pas besoin de one-vs-rest
multiclasse) : RandomForest/LogisticRegression via `probability` (Spark, colonne classe 1),
XGBoost/LightGBM via `predict_proba` (sklearn, sur `X_val` déjà en mémoire).

In [ ]:
from sklearn.metrics import roc_curve, auc
import matplotlib.pyplot as plt

plt.figure(figsize=(7, 6))

for nom_algo, modele in modeles_entraines.items():
    preds = modele.transform(df_val).select("label_idx", "probability").toPandas()
    proba_classe1 = np.vstack(preds["probability"].apply(lambda v: v.toArray()))[:, 1]
    fpr, tpr, _ = roc_curve(preds["label_idx"], proba_classe1)
    plt.plot(fpr, tpr, label=f"{nom_algo} (AUC={auc(fpr, tpr):.3f})")

for nom_algo, modele in [("XGBoost", xgb_model), ("LightGBM", lgbm_model)]:
    proba_classe1 = modele.predict_proba(X_val)[:, 1]
    fpr, tpr, _ = roc_curve(y_val, proba_classe1)
    plt.plot(fpr, tpr, label=f"{nom_algo} (AUC={auc(fpr, tpr):.3f})")

plt.plot([0, 1], [0, 1], linestyle="--", color="grey", linewidth=1)
plt.xlabel("Taux de faux positifs")
plt.ylabel("Taux de vrais positifs")
plt.title("Courbes ROC — comparaison des algorithmes (classe 1)")
plt.legend(fontsize=9)
plt.tight_layout()
plt.show()

In [ ]:
df = df_val

for i, stage in enumerate(modeles_entraines["RandomForest"].stages):
    try:
        df = stage.transform(df)
        print(f"Stage {i:2d} ({stage.__class__.__name__}) : OK")
    except Exception as e:
        print(f"Stage {i:2d} ({stage.__class__.__name__}) : FAILED")
        print(e)
        break

## 13. Sélection automatique + refit sur 100% du train & sauvegarde

Le vainqueur de `comparaison_finale` (F1 classe 1, section 11) est maintenant choisi
**automatiquement** parmi les 4 candidats réels — la comparaison MLlib/XGBoost/LightGBM tourne
désormais correctement (section 10/11), donc plus besoin de fixer `"XGBoost"` en dur comme
c'était le cas en V1.2 (fast path isolé, sans comparaison possible). Deux branches de
sauvegarde selon le type du vainqueur : `PipelineModel` complet pour MLlib, encodeur Spark +
modèle sklearn (`joblib`) séparés pour XGBoost/LightGBM.

In [ ]:
spark = get_spark()

In [ ]:
import joblib

nom_meilleur_modele = comparaison_finale.iloc[0]["algo"]
TYPE_MODELE_FINAL = "mllib" if nom_meilleur_modele in ("RandomForest", "LogisticRegression") else "sklearn"
print(f"Modèle retenu (F1 classe 1 le plus élevé) : {nom_meilleur_modele} ({TYPE_MODELE_FINAL})")

def construire_estimateur_final(nom):
    """CORRECTIF (bug silencieux) : modeles_entraines[nom].stages[-1] est un MODELE deja
    fitte (RandomForestClassificationModel / LogisticRegressionModel), pas l'Estimator
    d'origine. Pipeline.fit() ne re-fit QUE les stages qui sont des Estimator -- un stage
    deja fitte est simplement reutilise tel quel. Le "refit sur 100%" annonce par le print
    plus bas ne se produisait donc jamais pour le classifieur (seuls les StringIndexer/
    OneHotEncoder de l'encodage etaient reellement refits) : le modele final sauvegarde
    restait entraine sur les 80% de df_fit. On reconstruit ici un Estimator neuf avec les
    memes hyperparametres que la section 8, pour qu'il soit reellement refit.
    """
    if nom == "RandomForest":
        return RandomForestClassifier(
            labelCol="label_idx", featuresCol="features", predictionCol="prediction",
            probabilityCol="probability", weightCol="poids_classe",
            numTrees=50, maxDepth=8, minInstancesPerNode=1, maxBins=max_bins, seed=RANDOM_SEED,
        )
    else:
        return LogisticRegression(
            labelCol="label_idx", featuresCol="features", predictionCol="prediction",
            probabilityCol="probability", weightCol="poids_classe", family="multinomial",
            regParam=0.01, elasticNetParam=0.0,
        )


if TYPE_MODELE_FINAL == "mllib":
    # Refit du pipeline complet (encodage + classifieur) sur 100% de df_train_full, avec les
    # mêmes hyperparamètres que le modèle retenu (section 8) -- poids_classe recalculé dessus.
    dernier_stage = construire_estimateur_final(nom_meilleur_modele)
    poids_classe_full = calculer_poids_classe(df_train_full)
    df_train_full_pondere = df_train_full.join(poids_classe_full, on=COL_LABEL, how="left")
    pipeline_final = construire_pipeline_algo(dernier_stage)
    pipeline_model_final = pipeline_final.fit(df_train_full_pondere)
    pipeline_model_final.write().overwrite().save(MODEL_PATH)
    print(f"PipelineModel sauvegardé : {MODEL_PATH}")

else:
    seuil_final = SEUIL_XGB if nom_meilleur_modele == "XGBoost" else SEUIL_LGBM
    modele_sklearn_final = xgb_model if nom_meilleur_modele == "XGBoost" else lgbm_model

    # CORRECTIF (cause du crash "erreur pendant la compilation") : cet encodeur etait
    # reconstruit ici avec COLS_CATEGORIELLES_BASSE_CARDINALITE seul -- oubliant les 9
    # colonnes binnees et la colonne d'interaction ajoutees plus tard (section 6/17,
    # "MISE A JOUR"). Le pipeline ne generait donc plus les colonnes *_ohe attendues par
    # `assembler` (qui, lui, connait la vraie liste feature_cols_encodees) -> Spark levait
    # une AnalysisException (colonne introuvable) des que XGBoost/LightGBM gagnait la
    # comparaison (section 11) et declenchait cette branche. On reutilise `encodage_stages`,
    # la liste globale deja construite section 6 avec les BONNES colonnes (basse cardinalite
    # + binnees), au lieu d'en reconstruire une partielle ici.
    encodeur_full_pipeline = Pipeline(stages=encodage_stages + [label_indexer, assembler])
    encodeur_final = encodeur_full_pipeline.fit(df_train_full)

    pdf_full = encodeur_final.transform(df_train_full).select("features", "label_idx").toPandas()
    X_full = np.vstack(pdf_full["features"].apply(lambda v: v.toArray()))
    y_full = pdf_full["label_idx"].astype(int).values

    modele_sklearn_final.fit(X_full, y_full)  # mêmes hyperparamètres (best_params_), refit sur 100%

    encodeur_final.write().overwrite().save(MODEL_PATH + "_encodeur")
    joblib.dump(modele_sklearn_final, MODEL_PATH + "_sklearn.joblib")
    joblib.dump({"seuil_decision": seuil_final}, MODEL_PATH + "_meta.joblib")
    print(f"Encodeur Spark sauvegardé : {MODEL_PATH}_encodeur")
    print(f"Modèle sklearn sauvegardé : {MODEL_PATH}_sklearn.joblib")
    print(f"Seuil de décision sauvegardé ({seuil_final:.2f}) : {MODEL_PATH}_meta.joblib")

## 14. Scoring (`dataset_a_scorer`)

Chargement du modèle sauvegardé (branche MLlib ou sklearn selon `TYPE_MODELE_FINAL`, section 13)
et scoring de `dataset_a_scorer` — aucun `.fit()` ici, tout ce qui a été appris vient de
`df_train_full`.

In [ ]:
if PATH_SCORER_IN is not None:
    df_scorer = spark.read.parquet(PATH_SCORER_IN)

    if TYPE_MODELE_FINAL == "mllib":
        pipeline_model_reload = PipelineModel.load(MODEL_PATH)
        predictions = pipeline_model_reload.transform(df_scorer)

        label_indexer_model = pipeline_model_reload.stages[len(encodage_stages)]
        converter = IndexToString(inputCol="prediction", outputCol="label_predit", labels=label_indexer_model.labels)
        predictions = converter.transform(predictions)

        # Extract épargne probability (positive class, index 1) from probability vector
        from pyspark.sql import functions as F
        from pyspark.sql.window import Window
        
        predictions_with_score = predictions.withColumn(
            "epargne_score",
            F.element(F.col("probability"), 1)
        )

        # Add ranking: highest score = rank 1 (most promising)
        window_spec = Window.orderBy(F.desc("epargne_score"))
        predictions_with_rank = predictions_with_score.withColumn(
            "rank",
            F.row_number().over(window_spec)
        )

        # Select output columns: client_id (RADICAL), épargne_score, rank
        cols_output = [c for c in ["RADICAL"] if c in predictions_with_rank.columns] + ["epargne_score", "rank"]
        results_df = predictions_with_rank.select(*cols_output)
        
        results_df.show(20, truncate=False)
        results_df.write.mode("overwrite").parquet(PATH_PREDICTIONS_OUT)
        
    else:
        encodeur_reload = PipelineModel.load(MODEL_PATH + "_encodeur")
        modele_reload = joblib.load(MODEL_PATH + "_sklearn.joblib")
        meta_reload = joblib.load(MODEL_PATH + "_meta.joblib")
        # seuil_reload is no longer used for scoring (we want raw probabilities)
        # seuil_reload = meta_reload["seuil_decision"]

        cols_id = [c for c in ["RADICAL", "CODE_VILLE"] if c in df_scorer.columns]
        pdf_scorer = encodeur_reload.transform(df_scorer).select(*cols_id, "features").toPandas()
        X_scorer = np.vstack(pdf_scorer["features"].apply(lambda v: v.toArray()))

        # Get épargne probabilities (positive class)
        probas_scorer = modele_reload.predict_proba(X_scorer)[:, 1]
        pdf_scorer["epargne_score"] = probas_scorer
        
        # Add ranking: highest score = rank 1 (most promising)
        pdf_scorer["rank"] = pdf_scorer["epargne_score"].rank(ascending=False, method='first').astype(int)
        
        # Select output columns
        cols_output = [c for c in ["RADICAL"] if c in pdf_scorer.columns] + ["epargne_score", "rank"]
        pdf_output = pdf_scorer[cols_output]

        print(pdf_output.head(20))
        spark.createDataFrame(pdf_output).write.mode("overwrite").parquet(PATH_PREDICTIONS_OUT)

    print(f"Épargne scores written : {PATH_PREDICTIONS_OUT}")
else:
    print("PATH_SCORER_IN non defini (LOCAL_MODE=True, dataset_a_scorer pas encore teste en local) "
          "-- basculer LOCAL_MODE=False une fois pret pour le cluster complet.")

## 15. Limites de cette V1.3 & prochaines étapes

**Correctifs V1.2 → V1.3 (cette session)** :
- **Section 7-9 (V1.2) supprimée** — un 3ᵉ `RandomForest` d'exploration (hyperparamètres par
  défaut, `numTrees=30, maxDepth=5`) qui refaisait un `.fit()` complet pour rien : le
  `RandomForest` retenu (section 8) a déjà ses hyperparamètres définitifs.
- **Toute la mécanique `CrossValidator`/`algos_config`/`entrainer_un_algo` (section 9bis)
  supprimée** — elle ne sert qu'à *chercher* des hyperparamètres, déjà trouvés (`numTrees=50,
  maxDepth=8` / `regParam=0.01, elasticNetParam=0.0`). La relancer à chaque run coûtait des
  heures pour un résultat déjà connu ; section 8 fait maintenant un fit unique avec ces valeurs.
- **9quater (`RandomizedSearchCV` `n_jobs=1`, grille + petite, seuil balayé manuellement)
  remplacée par le fast path** (section 10) — la version qui faisait planter WSL après
  quelques folds (usage CPU/Docker anormal) est retirée ; XGBoost et LightGBM utilisent
  maintenant exactement le même chemin de code, celui qui a tourné sans incident ce matin
  (`n_jobs=2` borné, `scale_pos_weight` cherché, seuil vectorisé).
- **Checkpoints ajoutés pour les 4 algorithmes** — `RandomForest`/`LogisticRegression`
  (`PipelineModel`, MinIO) et `XGBoost`/`LightGBM` (`joblib` + métadonnées json, disque local),
  chacun contrôlé par son propre interrupteur (section 7). Plus besoin de tout réentraîner
  pour tester un seul modèle, et un restart de kernel ne coûte plus un refit complet.
- **`X_fit`/`X_val`/`y_fit`/`y_val` mis en cache disque** (section 9) — le `.toPandas()` sur la
  population complète (~10 min) ne se répète plus à chaque run.
- **Sélection automatique du modèle final restaurée** (section 13) — la comparaison à 4
  candidats fonctionne maintenant réellement (section 11), donc `nom_meilleur_modele` n'a plus
  besoin d'être fixé en dur.

**Hérité de la V1.2, toujours valable** :
- Tomek Links non appliqué ici (réservé à `dataset_produit`, cf. notebook EDA).
- Pondération par classe adoucie (`sqrt`) plutôt que le ratio brut.
- Seuil de décision optimisé sauvegardé et réappliqué au scoring (jamais `.predict()` seul).

**Pas encore fait, pistes pour la suite** :
- `SMOTENC` pour `CODE_VILLE_idx` (catégoriel) — pas implémenté, prudence sur le `.toPandas()`
  nécessaire.
- Cross-validation sur XGBoost/LightGBM limitée au `cv=3` interne de `RandomizedSearchCV` sur
  `X_fit`/`y_fit` — l'évaluation finale reste un split 80/20 unique.
- `CODE_VILLE_idx` reste un entier arbitraire, acceptable pour des modèles à base d'arbres
  (les 4 candidats ici), à revoir si un modèle linéaire non pénalisé par un ordre arbitraire
  entre en jeu.
- `MODEL_PATH` ne gère qu'un seul modèle, pas de versionnement.